# Ito & Doya (2015) on dandiset 001642

**Paper.** Ito M, Doya K (2015). Distinct neural representation in the dorsolateral,
dorsomedial, and ventral parts of the striatum during fixed- and free-choice tasks. J Neurosci
35(8):3499-3514. doi:10.1523/jneurosci.1962-14.2015.

**Dataset.** DANDI:001642, version 0.260205.0153 (doi:10.48324/dandi.001642/0.260205.0153),
streamed from the archive. The paper cites no version of the dandiset (it predates it), and
0.260205.0153 is the only published version; the draft has the same 87 assets and size, so the
published version is pinned.

**Claims.** `claims/001642/10.1523-jneurosci.1962-14.2015.yaml`, written from the version of
record (PMC4339358): main text, Methods and figure legends; the display equations and the
figure panels could not be read, and the paper links no code.

All 32 claims are analysed here. Two end blocked after the analysis, as their readings
disagree and the paper gives no basis to choose (fig5g-reward-info, fig10c-vs-no-fixed-free-
difference); the summary at the end gives every outcome.

| Claim | Section | Status |
|---|---|---|
| trial-counts, trials-per-block | behaviour | tested |
| fig2c-extinction-differs, fig2b-extinction-time-course | extinction tests | tested |
| fig2d-f-decision-trees | decision trees | tested |
| fig2g-fq-variable-best, actor-critic-near-chance | choice models | tested |
| neuron-counts | units | tested |
| fig4b-vs-phase1, fig4b-dms-phase5, fig4b-vs-phase7, fig4c-activation-duration | EASH | tested |
| fig5b-state-info-vs, fig5e-action-info-rises-dls, fig5e-action-info-cue-offset-dls, fig5e-action-info-center-exit, fig5d-action-info-phase5-dms | mutual information | tested |
| fig5g-reward-info | mutual information | tested; blocked (methods_underspecified) |
| fig6-event-coding-proportions, fig6-above-chance, fig6bc-similar-fixed-free | event coding | tested |
| fig8a-d-action-reward-coding, fig8e-action-value-dms, fig8e-qr-dms, fig8f-state-value-vs, no-policy-coding | regression | tested |
| fig9-temporal-order-all-blocks, fig9-dms-never-above-dls | MI by block type | tested |
| fig10a-dls-fixed-above-free, fig10b-dms-fixed-above-free, fig10d-dls-stay-above-switch | MI by block type | tested |
| fig10c-vs-no-fixed-free-difference | MI by block type | tested; blocked (methods_underspecified) |

Loading and the measures are in `striatum.py`, the choice models and the regression in
`models.py`; their docstrings say where each parameter comes from. The whole notebook takes
about half an hour, most of it the shuffles of the mutual information and the particle
filters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

import striatum as S

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
SEED = 20150225
print("dandiset", S.DANDISET, "version", S.VERSION)

In [ ]:
D = S.load_all()
T = pd.concat([d["trials"] for d in D], ignore_index=True)
B = pd.concat([d["blocks"] for d in D], ignore_index=True)
U = pd.concat([d["units"] for d in D], ignore_index=True)
REC = sorted(U.session.unique())  # sessions that hold units
print(len(D), "files;", len(REC), "with units;", len(T), "trials;", len(U), "units")

## Data check

What was loaded, and the checks of the descriptions against the values.

In [ ]:
print("Files per subject:", T.groupby("subject").session.nunique().to_dict())
print("Sessions without units:", sorted(set(T.session) - set(REC)))
print("\nTrials by block type (from the block's reward rule) and result code:")
print(pd.crosstab(T.btype, T.result, margins=True))
print("\nTrials by block type and state code:")
print(pd.crosstab(T.btype, T.state))
print("\nBlocks by reward rule (L, R):")
print(B.groupby(["RewardRule_L", "RewardRule_R"]).size())
print("\nUnits by area and deposited PANs flag:")
print(pd.crosstab(U.Area, U.PANs, margins=True))

In [ ]:
# The trial table says state 1 = Tone A (left), 2 = Tone B (right), 3 = Tone C (choice).
# The reward contingencies say otherwise: which (state, action) pairs are ever rewarded, and
# in which blocks each state occurs.
ch = T[T.choice]
print(pd.crosstab([ch.state, ch.action], ch.reward))
print("\nState codes in single fixed-choice blocks with rule (50, 0) and (0, 50):")
sf = ch[ch.btype == "SF"]
print(pd.crosstab([sf.rule_L, sf.rule_R], sf.state))
print("\nFraction of left choices per state in SF, DF and FREE blocks:")
print(ch[ch.btype != "EXT"].groupby(["btype", "state"]).action.apply(lambda a: (a == 1).mean()).round(3))

In [ ]:
# Event times: order and phase durations against the paper's means (Methods, EASH).
ok = T[T.events_ok]
ev = ok[S.EVENT_COLS].to_numpy()
dur = np.diff(ev, axis=1)
print("Choice trials:", T.choice.sum(), "; with all six events in order:", len(ok),
      f"({T.choice.sum() - len(ok)} dropped)")
print(pd.DataFrame({"paper mean (s)": S.PHASE_DUR[1:6], "data mean": dur.mean(0).round(3),
                    "data median": np.median(dur, 0).round(3), "1st pct": np.percentile(dur, 1, 0).round(3),
                    "99th pct": np.percentile(dur, 99, 0).round(2), "min": dur.min(0).round(3),
                    "max": dur.max(0).round(2)}, index=[f"Phase {i}" for i in range(2, 7)]))
print("\nCue duration (E2 to E3) range, s:", dur[:, 1].min().round(3), dur[:, 1].max().round(3),
      "(paper 1 to 2 s)")
print("Reward times present on rewarded choice trials:",
      T[T.choice & (T.reward == 1)].reward_on_time.notna().mean().round(3),
      "; on unrewarded:", T[T.choice & (T.reward == 0)].reward_on_time.notna().mean().round(3))
print("Unit spike counts: min", U.n_spikes.min(), "median", int(U.n_spikes.median()), "max", U.n_spikes.max())
st_sorted = all(np.all(np.diff(s) >= 0) for s in U.spikes if len(s) > 1)
print("Spike times sorted in every unit:", st_sorted)
lo = min(s[0] for s in U.spikes if len(s)); hi = max(s[-1] for s in U.spikes if len(s))
print(f"Spike times range {lo:.2f} to {hi:.1f} s; trial starts {T.start_time.min():.2f} to {T.start_time.max():.1f} s")

## trial-counts

> The analysis covers 78,107 trials in 86 recording sessions from seven rats, of which 12,185
> are single fixed-choice, 22,234 double fixed-choice, 36,292 free-choice and 3505
> extinction-test trials.

Readings (criterion): (a) all trials started, including wait-error and error trials; (b)
trials with an L/R choice only. Counted over all 87 files and over the 86 sessions with units.

In [ ]:
paper = {"total": 78107, "SF": 12185, "DF": 22234, "FREE": 36292, "EXT": 3505}
rows = []
for scope, tt in [("87 files", T), ("86 sessions with units", T[T.session.isin(REC)])]:
    for reading, sel in [("(a) all trials", tt), ("(b) choice trials", tt[tt.choice])]:
        c = sel.btype.value_counts()
        r = {"scope": scope, "reading": reading, "sessions": sel.session.nunique(),
             "rats": sel.subject.nunique(), "total": len(sel)}
        r.update({k: int(c.get(k, 0)) for k in ["SF", "DF", "FREE", "EXT"]})
        r["within 10%"] = all(abs(r[k] - v) <= 0.1 * v for k, v in paper.items()) and \
            abs(r["sessions"] - 86) <= 8.6 and r["rats"] == 7
        rows.append(r)
tc = pd.DataFrame(rows)
print(tc.to_string())
print("\nPaper:", paper, "; the paper's four categories sum to", sum(v for k, v in paper.items() if k != "total"))
print("Wait-error (result 2) and other non-choice trials:", int((~T.choice).sum()))

**Result: reproduced.** Counting trials with an L/R choice in the 86 sessions that hold units (reading b), the dataset has 75,348 trials from 7 rats: 12,639 single fixed-choice, 22,695 double fixed-choice, 36,496 free-choice and 3518 extinction trials, all within 10% of the paper's numbers. Reading (a), all 79,431 trials started, puts the single fixed-choice count (13,485) outside 10%. The 87th file (LE115_14) holds behaviour without units and is left out of the 86 recording sessions.

## trials-per-block

> Completing a block took 22.56 ± 8.70 trials for single fixed-choice blocks, 47.11 ± 20.29
> for double fixed-choice blocks and 41.10 ± 27.58 for free-choice blocks (mean ± SD).

Readings (criterion): (a) choice trials, completed blocks; (b) all trials, completed blocks;
(c) as (a) with unfinished blocks; (d) as (b) with them. Completed is `FinishedBlock` = 1.

In [ ]:
bt = T.groupby(["session", "block"]).agg(btype=("btype", "first"), n_all=("trial", "size"),
                                          n_choice=("choice", "sum")).reset_index()
bt = bt.merge(B[["session", "block", "FinishedBlock"]], on=["session", "block"])
pp = {"SF": (22.56, 8.70), "DF": (47.11, 20.29), "FREE": (41.10, 27.58)}
rows = []
for reading, col, fin in [("(a)", "n_choice", True), ("(b)", "n_all", True), ("(c)", "n_choice", False), ("(d)", "n_all", False)]:
    sel = bt[bt.FinishedBlock == 1] if fin else bt
    ok_all = True
    for k, (m, sd) in pp.items():
        x = sel.loc[sel.btype == k, col]
        okm = abs(x.mean() - m) <= sd
        oks = abs(x.std(ddof=1) - sd) <= 0.1 * sd
        ok_all &= okm and oks
        rows.append({"reading": reading, "block": k, "n blocks": len(x), "mean": round(x.mean(), 2),
                     "SD": round(x.std(ddof=1), 2), "paper": f"{m} ± {sd}", "mean ok": okm, "SD ok": oks})
    print(reading, "all six within tolerance:", ok_all)
print(pd.DataFrame(rows).to_string())
print("Unfinished blocks:", int((bt.FinishedBlock == 0).sum()), "of", len(bt))

**Result: reproduced.** Counting choice trials in completed blocks (reading a), blocks take 22.50 ± 8.62 (single fixed, 551 blocks), 47.15 ± 20.34 (double fixed, 469) and 41.05 ± 27.54 (free, 866) trials, against the paper's 22.56 ± 8.70, 47.11 ± 20.29 and 41.10 ± 27.58. Readings (b) and (c) also match; reading (d) misses the double fixed-choice SD (22.86).

## Extinction tests (fig2c-extinction-differs, fig2b-extinction-time-course)

Extinction blocks are the blocks with reward rule (0, 0). Their trials are numbered by the
order of choice trials within the block (wait-error trials, which have no choice, are left
out). For a choice-tone extinction block the optimal action of the previous free-choice block
(the side with the higher reward probability) is taken from that block's reward rule
(Fig. 2B legend).

In [ ]:
ext = T[(T.btype == "EXT") & T.choice].copy()
ext["k"] = ext.groupby(["session", "block"]).cumcount() + 1
Bi = B.set_index(["session", "block"])
def prev_optimal(ses, blk):
    b = blk - 1
    while b >= 1:
        l, r = Bi.loc[(ses, b), ["RewardRule_L", "RewardRule_R"]]
        if (l, r) not in [(0, 0), (50, 0), (0, 50), (50, 50)]:
            return "L" if l > r else "R"
        b -= 1
    return None
eb = ext.groupby(["session", "block"]).agg(cue=("cue", "first"), n=("k", "size")).reset_index()
eb["prev_opt"] = [prev_optimal(s, b) if c == "choice" else None for s, b, c in zip(eb.session, eb.block, eb.cue)]
ext = ext.merge(eb[["session", "block", "prev_opt"]], on=["session", "block"])
ext["group"] = np.where(ext.cue == "choice", "choice after " + ext.prev_opt.astype(str) + "-optimal", ext.cue + " tone")
print("Extinction blocks by cue:", eb.cue.value_counts().to_dict(), "; choice trials per block:", eb.n.value_counts().to_dict())
print("Choice-tone blocks by previous optimal action:", eb.prev_opt.value_counts().to_dict())
ext5 = ext[ext.k <= 5]
tab = ext5.groupby("group").action.agg(left=lambda a: (a == 1).sum(), right=lambda a: (a == 2).sum())
tab["P(left)"] = (tab.left / (tab.left + tab.right)).round(3)
print(tab)

def chi(t, yates):
    c, p, _, _ = stats.chi2_contingency(np.asarray(t), correction=yates)
    return c, p
rows = []
for yates in (False, True):
    # reading (a): left tone vs choice after left-optimal; right tone vs choice after right-optimal
    for fixed, choice in [("left tone", "choice after L-optimal"), ("right tone", "choice after R-optimal")]:
        t = tab.loc[[fixed, choice], ["left", "right"]]
        c, p = chi(t, yates)
        pf, pc = tab.loc[fixed, "P(left)"], tab.loc[choice, "P(left)"]
        rows.append({"reading": "(a) " + fixed, "Yates": yates, "chi2": round(c, 1), "p": p,
                     "fixed P(left)": pf, "choice P(left)": pc,
                     "fixed further from 0.5": abs(pf - .5) > abs(pc - .5)})
    # reading (b): previously optimal action, fixed tones pooled vs choice tone pooled
    e = ext5.copy()
    e["opt"] = np.where(e.cue == "left", 1, np.where(e.cue == "right", 2, np.where(e.prev_opt == "L", 1, 2)))
    e["chose_opt"] = e.action == e.opt
    e["fixed"] = e.cue != "choice"
    t = pd.crosstab(e.fixed, e.chose_opt)
    c, p = chi(t, yates)
    po = e.groupby("fixed").chose_opt.mean()
    rows.append({"reading": "(b) pooled", "Yates": yates, "chi2": round(c, 1), "p": p,
                 "fixed P(left)": f"P(opt) {po[True]:.3f}", "choice P(left)": f"P(opt) {po[False]:.3f}",
                 "fixed further from 0.5": abs(po[True] - .5) > abs(po[False] - .5)})
res2c = pd.DataFrame(rows)
print(res2c.to_string())
print("Every test p < 0.0001 with the fixed tone further from 0.5:",
      bool((res2c.p < 1e-4).all() and res2c["fixed further from 0.5"].all()))
print("By rat, reading (b) P(optimal):")
print(e.groupby(["subject", "fixed"]).chose_opt.mean().unstack().round(3))

**Result: reproduced.** Over the five extinction trials the left tone gives P(left) 0.822 and the right tone 0.108, against 0.646 and 0.344 for the choice tone after left- and right-optimal blocks. Every chi-square test gives p below 1e-15 (largest 2.4e-16), with and without Yates' correction, and the fixed tones are further from 0.5 in every test. The pooled P(optimal) is higher for the fixed tones in all seven rats.

## fig2b-extinction-time-course

> In extinction tests the choice-tone choice probability shifts toward 50% after the first
> unrewarded trial, while the left- and right-tone choice probabilities stay biased after five
> unrewarded trials.

In [ ]:
rows = []
for g, gg in ext5.groupby("group"):
    for k, kk in gg.groupby("k"):
        nl = int((kk.action == 1).sum()); n = len(kk)
        w = proportion_confint(nl, n, 0.05, "wilson"); cp = proportion_confint(nl, n, 0.05, "beta")
        rows.append({"group": g, "trial": k, "n": n, "P(left)": nl / n, "wilson lo": w[0], "wilson hi": w[1],
                     "CP lo": cp[0], "CP hi": cp[1]})
tb = pd.DataFrame(rows).round(3)
print(tb.to_string())
fig, ax = plt.subplots(figsize=(5, 3.2))
for g, gg in tb.groupby("group"):
    ax.plot(gg.trial, gg["P(left)"], marker="o", label=g)
    ax.fill_between(gg.trial, gg["wilson lo"], gg["wilson hi"], alpha=.2)
ax.set_xlabel("extinction trial"); ax.set_ylabel("P(left)"); ax.set_ylim(0, 1); ax.legend(fontsize=7)
plt.show()
tbi = tb.set_index(["group", "trial"])
for ci in ["wilson", "CP"]:
    c1 = all(abs(tbi.loc[(g, 2), "P(left)"] - .5) < abs(tbi.loc[(g, 1), "P(left)"] - .5)
             for g in ["choice after L-optimal", "choice after R-optimal"])
    lo = "wilson lo" if ci == "wilson" else "CP lo"; hi = lo.replace("lo", "hi")
    c2 = tbi.loc[("left tone", 5), lo] > .5 and tbi.loc[("right tone", 5), hi] < .5
    print(f"{ci}: (1) choice tone closer to 0.5 at trial 2 than 1: {c1}; (2) trial-5 intervals exclude 0.5 on the biased side: {c2}")

**Result: reproduced.** For the choice tone P(left) moves from 0.815 to 0.551 (after left-optimal blocks) and from 0.175 to 0.379 (after right-optimal blocks) between the first and second extinction trials. At trial 5 the left tone is at 0.818 (Wilson interval 0.765 to 0.862) and the right tone at 0.155 (0.115 to 0.207); the Clopper-Pearson intervals give the same.

## fig2d-f-decision-trees

> Choice-tone choices depend on the previous one or two trials' actions and rewards (a
> decision tree that expands symmetrically), while left- and right-tone choices stay biased
> toward left and right whatever the previous experience.

Experiences e(t) = (action, reward) of the previous trials of the same tone within a session
(Methods, Decision trees). Readings: sequences running across block borders or restarted at
each block; for the choice tone, free-choice trials only, or with the choice-tone extinction
trials in the sequence too (the Methods say "choice-tone trials" without saying whether
extinction trials are among them). Left- and right-tone trees use single and double
fixed-choice blocks. 95% intervals Wilson and Clopper-Pearson.

In [ ]:
def tree(tt, across_blocks):
    """Counts of left and right choices after each experience sequence (d = 1, 2)."""
    rows = []
    for (ses,), g in tt.groupby(["session"]):
        keys = g.block.to_numpy() if not across_blocks else np.zeros(len(g))
        a = np.where(g.action.to_numpy() == 1, "L", "R"); r = g.reward.to_numpy().astype(int)
        e = np.array([f"{x}{y}" for x, y in zip(a, r)])
        for t in range(len(g)):
            rows.append({"d0": "", "a": a[t],
                         "d1": e[t - 1] if t >= 1 and keys[t - 1] == keys[t] else None,
                         "d2": e[t - 2] if t >= 2 and keys[t - 2] == keys[t] else None})
    df = pd.DataFrame(rows)
    out = []
    for name, keyc in [("d=1", ["d1"]), ("d=2", ["d1", "d2"])]:
        sub = df.dropna(subset=keyc)
        for k, gg in sub.groupby(keyc):
            k = k if isinstance(k, tuple) else (k,)
            nl = int((gg.a == "L").sum()); n = len(gg)
            w = proportion_confint(nl, n, 0.05, "wilson"); cp = proportion_confint(nl, n, 0.05, "beta")
            out.append({"d": name, "experience": " ".join(k), "n": n, "P(L)": nl / n,
                        "wil lo": w[0], "wil hi": w[1], "CP lo": cp[0], "CP hi": cp[1]})
    return pd.DataFrame(out)

ch = T[T.choice].sort_values(["session", "trial"])
choice_sets = {"free only": ch[(ch.cue == "choice") & (ch.btype == "FREE")],
               "free + extinction": ch[ch.cue == "choice"]}
fixed_sets = {c: ch[(ch.cue == c) & ch.btype.isin(["SF", "DF"])] for c in ["left", "right"]}
verdict = []
for across in (True, False):
    for cname, cs in choice_sets.items():
        tr = tree(cs, across).set_index(["d", "experience"])
        for ci in ("wil", "CP"):
            L1, R1 = tr.loc[("d=1", "L1")], tr.loc[("d=1", "R1")]
            c1 = (L1["P(L)"] > R1["P(L)"]) and (L1[f"{ci} lo"] > R1[f"{ci} hi"])
            verdict.append({"across blocks": across, "choice set": cname, "CI": ci, "part": "(1)",
                            "holds": c1, "detail": f"P(L|L1)={L1['P(L)']:.3f} [{L1[f'{ci} lo']:.3f},{L1[f'{ci} hi']:.3f}], P(L|R1)={R1['P(L)']:.3f} [{R1[f'{ci} lo']:.3f},{R1[f'{ci} hi']:.3f}]"})
        if across and cname == "free only":
            print("Choice-tone tree (free-choice trials, sequences across blocks):")
            print(tr.round(3).to_string())
    trees = {c: tree(fs, across) for c, fs in fixed_sets.items()}
    for floor in (0, 10):
        lt = trees["left"][trees["left"].n >= floor]; rt = trees["right"][trees["right"].n >= floor]
        c2 = bool((lt["P(L)"] > .5).all() and (rt["P(L)"] < .5).all())
        verdict.append({"across blocks": across, "choice set": "-", "CI": "-", "part": f"(2) floor {floor}",
                        "holds": c2, "detail": f"left tone min P(L) {lt['P(L)'].min():.3f} over {len(lt)} branches; right tone max P(L) {rt['P(L)'].max():.3f} over {len(rt)}"})
    if across:
        for c in ["left", "right"]:
            print(f"\n{c}-tone tree (SF and DF, across blocks):")
            print(trees[c].round(3).to_string())
vd = pd.DataFrame(verdict)
print(vd.to_string())
print("All readings hold:", bool(vd.holds.all()))

**Result: reproduced.** For the choice tone P(L) after L1 is 0.804 (0.796 to 0.812) and after R1 0.185 (0.177 to 0.193). Every left-tone branch has P(L) of 0.715 or more and every right-tone branch 0.247 or less (sequences across blocks; 0.699 and 0.254 within blocks). All 12 readings hold.

## Choice models (fig2g-fq-variable-best, actor-critic-near-chance)

Choice-tone trials of free-choice blocks with an L/R choice, as one sequence per session
(the values carry across the free-choice blocks of a session; extinction trials are left
out). The dandiset has 86 sessions with free-choice trials, where the paper's comparison used
202 (101 training, 101 test); the others are not in the dandiset. Training and test halves:
(alt) alternate sessions in name order, (rand) a seeded random half. Models and their forms:
see the docstring of `models.py`. Normalized likelihood of test data per model, overall and
per test session; paired Wilcoxon signed-rank tests of each model against FQ (variable)
across test sessions (Fig. 2G legend).

In [ ]:
import models as M
fc = T[(T.btype == "FREE") & T.choice].sort_values(["session", "trial"])
SESS = {s: ((g.action == 1).to_numpy(), (g.reward == 1).to_numpy()) for s, g in fc.groupby("session")}
names = sorted(SESS)
rng_split = np.random.default_rng(SEED + 2)
perm = rng_split.permutation(len(names))
SPLITS = {"alt": ([names[i] for i in range(0, len(names), 2)], [names[i] for i in range(1, len(names), 2)]),
          "rand": ([names[i] for i in sorted(perm[:len(names) // 2])], [names[i] for i in sorted(perm[len(names) // 2:])])}
print(len(names), "sessions with free-choice trials;", sum(len(v[0]) for v in SESS.values()), "trials")
GRID_A = [0.0, 0.01, 0.05]
GRID_K = [0.0, 0.05, 0.2]

def run_models(split, seed):
    rng = np.random.default_rng(seed)
    tr_names, te_names = SPLITS[split]
    train = [SESS[s] for s in tr_names]; test = [SESS[s] for s in te_names]
    ll = {}; fitted = {}
    for d in (1, 2, 3, 4):
        cnt = M.markov_fit(d, train)
        ll[f"Markov {d}"] = [M.markov_loglik(d, cnt, a, r) for a, r in test]
    for kind in ("Q", "FQ", "DFQ"):
        x, _ = M.fit_const(kind, train, rng)
        fitted[f"{kind} const"] = np.round(x, 3)
        ll[f"{kind} const"] = [M.q_loglik(x, kind, a, r) for a, r in test]
    for kind in ("Q", "FQ", "DFQ"):
        best = None
        for sa in GRID_A:
            for sk in GRID_K:
                L = sum(M.pf_session(kind, a, r, sa, sk, rng).sum() for a, r in train)
                if best is None or L > best[0]:
                    best = (L, sa, sk)
        fitted[f"{kind} variable"] = best[1:]
        ll[f"{kind} variable"] = [M.pf_session(kind, a, r, best[1], best[2], rng) for a, r in test]
    for shared in (True, False):
        nm = "actor-critic " + ("(a) shared rate" if shared else "(b) separate rates")
        x, _ = M.fit_const(None, train, rng, loglik=lambda p, a, r, s=shared: M.ac_loglik(p, s, a, r), bounds=M.AC_BOUNDS[shared])
        fitted[nm] = np.round(x, 3)
        ll[nm] = [M.ac_loglik(x, shared, a, r) for a, r in test]
    return ll, fitted

MODEL_RES = {sp: run_models(sp, SEED + 20 + i) for i, sp in enumerate(SPLITS)}

In [ ]:
rows = []
for sp, (ll, fitted) in MODEL_RES.items():
    ref = np.array([M.norm_lik(x) for x in ll["FQ variable"]])
    for m, lls in ll.items():
        per = np.array([M.norm_lik(x) for x in lls])
        p = stats.wilcoxon(per, ref).pvalue if m != "FQ variable" else np.nan
        rows.append({"split": sp, "model": m, "normalized likelihood": M.norm_lik(np.concatenate(lls)),
                     "median per session": np.median(per), "Wilcoxon p vs FQ variable": p,
                     "fitted": fitted.get(m, "")})
r2g = pd.DataFrame(rows)
print(r2g.round(4).to_string())
for sp in SPLITS:
    d = r2g[(r2g.split == sp) & ~r2g.model.str.startswith("actor")]
    best = d.loc[d["normalized likelihood"].idxmax(), "model"]
    fqv = d.loc[d.model == "FQ variable", "normalized likelihood"].item()
    mk = d[d.model.str.startswith("Markov")]["normalized likelihood"].max()
    print(f"{sp}: best model {best}; FQ variable {fqv:.4f}; best Markov {mk:.4f}; FQ variable above every Markov: {fqv > mk}")

**Result: not reproduced.** On held-out sessions FQ learning with time-varying parameters reaches a normalized likelihood of 0.5758 (alternate split) and 0.5762 (random split), below DFQ variable (0.5770, 0.5775) and below the Markov models of order 3 (0.5770, 0.5782) and 4 (0.5785, 0.5797). It is above Q and FQ with constant parameters and Markov 1. All models are close: the range from Markov 2 to Markov 4 is 0.573 to 0.580. The model equations are missing from the paper's text and are taken as the claims file reads them (O8), the priors of the particle filter are assumed, and the comparison uses 86 sessions where the paper used 202.

In [ ]:
ac = r2g[r2g.model.str.startswith("actor")][["split", "model", "normalized likelihood", "fitted"]].copy()
ac["within 0.5080 ± 0.0508"] = (ac["normalized likelihood"] - 0.5080).abs() <= 0.0508
print(ac.round(4).to_string())

**Result: reproduced.** The regular actor-critic with constant parameters gives a normalized likelihood of 0.5000 (shared rate) and 0.5013 (separate rates) on the alternate split, and 0.5026 and 0.5000 on the random split, all within 0.0508 of the paper's 0.5080.

## neuron-counts

> Stable recordings were made from 260 DLS, 178 DMS and 179 VS neurons, of which 190, 105 and
> 119 were phasically active neurons.

The dandiset holds 859 units with an `Area` and a deposited `PANs` flag, so it is not only
the curated units. The criterion applies the PAN rule of the Methods (proportion of time in
ISIs longer than 1 s greater than 0.4, whole session). Of the five exclusion conditions of
the Methods (Electrophysiological recording), (1), (2) and (5) need waveforms or histology,
which are not in the files (CS units are left out as outside the three targets). Conditions
(3) and (4) and the duplicate check are computed here from the spikes, as the Methods give
them: (3) PETHs from -4 to 4 s in 100 ms bins around cue onset, centre-hole exit and L/R entry
all below 1 Hz; (4) the EASH in 10 ms bins smoothed with a 10 ms Gaussian below 1 Hz "at any
time bin", read as (i) in every bin, (ii) in some bin; duplicates: cross-correlogram in 1 ms
bins between units on different wires of a bundle with the 0 ms bin 10 times the mean of
-200 to 200 ms (the PETH-similarity part of that rule is not defined, so it is not applied).

In [ ]:
allu = S.neurons(D, lambda u: True)
rows = []
peth_off = np.column_stack([np.arange(-4, 4, .1), np.arange(-4, 4, .1) + .1])
for n in allu:
    s = n["spikes"]; isi = np.diff(s)
    prop = isi[isi > 1].sum() / isi.sum()
    peth_max = max(S.rcounts(n, ev, peth_off).mean(0).max() / .1 for ev in (2, 4, 5))
    e10 = S.gauss_smooth(S.wcounts(n, S.bins(.01)).mean(0) / .01, 1)
    rows.append(dict(region=n["region"], session=n["session"], subject=n["subject"], unit=n["unit"],
                     prop=prop, peth_max=peth_max, eash_max=e10.max(), eash_min=e10.min()))
UN = pd.DataFrame(rows)
UN = UN.merge(U[["session", "Unit_Name", "PANs", "Electrode_ID"]].rename(columns={"Unit_Name": "unit"}), on=["session", "unit"])
UN["c3"] = UN.peth_max < 1.0
UN["c4i"] = UN.eash_max < 1.0
UN["c4ii"] = UN.eash_min < 1.0

# duplicate check within a bundle (wires 1-8, 9-16, 17-24, 25-32), different wires
UN["bundle"] = (UN.Electrode_ID - 1) // 8
spk = {(n["session"], n["unit"]): n["spikes"] for n in allu}
dup = set(); npairs = 0
for (ses, b), g in UN.groupby(["session", "bundle"]):
    g = g.sort_values("unit").to_dict("records")
    for i in range(len(g)):
        for j in range(i + 1, len(g)):
            if g[i]["Electrode_ID"] == g[j]["Electrode_ID"]:
                continue
            npairs += 1
            a, bb = spk[(ses, g[i]["unit"])], spk[(ses, g[j]["unit"])]
            zero = (np.searchsorted(bb, a + .0005) - np.searchsorted(bb, a - .0005)).sum()
            tot = (np.searchsorted(bb, a + .2005) - np.searchsorted(bb, a - .2005)).sum()
            if zero > 10 * max(tot - zero, 1) / 400:
                dup.add((ses, g[j]["unit"]))
UN["dup"] = [(s, u) in dup for s, u in zip(UN.session, UN.unit)]
UN["pan_rule"] = UN.prop > 0.4
print("Pairs checked:", npairs, "; units removed as duplicates:", len(dup))
paper = {"DLS": (260, 190), "DMS": (178, 105), "VS": (179, 119)}
rows = []
for name, keep in [("all units", np.ones(len(UN), bool)),
                   ("drop (3), (4i), duplicates", ~UN.c3 & ~UN.c4i & ~UN.dup),
                   ("drop (3), (4ii), duplicates", ~UN.c3 & ~UN.c4ii & ~UN.dup)]:
    for reg in S.REGIONS:
        m = keep & (UN.region == reg)
        st, pn = int(m.sum()), int((m & UN.pan_rule).sum())
        rows.append({"stable set": name, "region": reg, "stable": st, "paper stable": paper[reg][0],
                     "PANs (rule)": pn, "paper PANs": paper[reg][1],
                     "PANs (deposited flag)": int((m & (UN.PANs == 1)).sum()),
                     "stable ok": abs(st - paper[reg][0]) <= .1 * paper[reg][0],
                     "PAN ok": abs(pn - paper[reg][1]) <= .1 * paper[reg][1]})
nc = pd.DataFrame(rows)
print(nc.to_string())
print("\nDeposited flag against the rule (all DLS/DMS/VS units):")
print(pd.crosstab(UN.PANs, UN.pan_rule))
print("Rule-PANs without the deposited flag: median whole-session rate",
      round(float((UN[UN.pan_rule & (UN.PANs == 0)].eash_max).median()), 2), "Hz peak EASH;",
      "with the flag:", round(float(UN[UN.PANs == 1].eash_max.median()), 2), "Hz")
print("Deposited PANs failing (3):", int((UN.c3 & (UN.PANs == 1)).sum()), "; failing (4i):", int((UN.c4i & (UN.PANs == 1)).sum()))
print("\nBy rat, deposited PANs per region:")
print(pd.crosstab(UN[UN.PANs == 1].subject, UN[UN.PANs == 1].region))

**Result: not reproduced.** The dandiset holds 297 DLS, 255 DMS and 215 VS units; after the two exclusion conditions that can be computed and the duplicate check, 233, 204 and 142, against the paper's 260, 178 and 179 stable neurons. With the Methods' PAN rule, 183 DLS, 147 DMS and 119 VS PANs (DLS and VS within 10% of 190 and 119, DMS not). The deposited `PANs` flag gives 204, 112 and 138 (VS 16% above 119). No set matches all six counts. The deposited flag is a subset of the units that pass the ISI rule (454 of 628), so the flag rests on a further criterion that the dandiset does not describe.

## Unit sets for the neural claims

The analyses use the units the dandiset flags as PANs (`PANs` = 1) in DLS, DMS and VS: this
is the depositors' classification, and its counts (204, 112, 138) are the closest to the
paper's 190, 105 and 119 (see neuron-counts). Since the Methods' own rule gives a different
set, the main neural results are also shown on the set rebuilt from the Methods (rule PANs
after conditions (3), (4i) and the duplicate check: 183, 147, 119).

In [ ]:
PAN = S.neurons(D, lambda u: u.PANs == 1)
_keep = set(zip(UN.session[~UN.c3 & ~UN.c4i & ~UN.dup & UN.pan_rule], UN.unit[~UN.c3 & ~UN.c4i & ~UN.dup & UN.pan_rule]))
PAN_RULE = [n for n in allu if (n["session"], n["unit"]) in _keep]
SETS = {"deposited PANs": PAN, "Methods-rule PANs": PAN_RULE}
for k, v in SETS.items():
    print(k, pd.Series([n["region"] for n in v]).value_counts().reindex(S.REGIONS).to_dict())
REG = {k: np.array([n["region"] for n in v]) for k, v in SETS.items()}

def trial_mask(n, ext):
    """Trials of a neuron's session used for EASHs and Fig. 5 (O3): with or without extinction."""
    return np.ones(len(n["tr"]), bool) if ext else (n["tr"].btype != "EXT").to_numpy()

## Fig. 4 claims (fig4b-vs-phase1, fig4b-dms-phase5, fig4b-vs-phase7, fig4c-activation-duration)

Per PAN, the EASH (10 ms bins) over all choice trials, with (O3 b) or without (O3 a)
extinction trials, smoothed as O2 (10 ms SD, 30 ms SD, none), normalized to a maximum of 1.
Activated in a phase: (a) some bin at or above 0.5 in that phase; (b) the first crossing of 0.5
falls in that phase. Activity ratio (Fig. 4C legend): the fraction of a phase's bins above 0.5,
averaged over the seven phases. Chi-square tests are 2 x 2 (O4: without and with Yates'
correction); Mann-Whitney U tests two-sided.

In [ ]:
E10 = S.bins(0.01)
PH_OF_BIN = np.searchsorted(S.EDGES, E10[:, 0], "right") - 1  # 0..6 for Phases 1..7
def eash_all(nset, ext):
    return np.array([S.wcounts(n, E10)[trial_mask(n, ext)].mean(0) / 0.01 for n in nset])
EASH = {(k, ext): eash_all(v, ext) for k, v in SETS.items() for ext in (False, True)}

def fig4_measures(E, sd):
    sm = np.array([S.gauss_smooth(e, sd) for e in E]) if sd else E
    nrm = sm / sm.max(axis=1, keepdims=True)
    above = nrm >= 0.5
    any_ph = np.column_stack([above[:, PH_OF_BIN == p].any(1) for p in range(7)])
    first = PH_OF_BIN[np.argmax(above, axis=1)]
    first_ph = np.column_stack([first == p for p in range(7)])
    ratio = np.column_stack([(nrm[:, PH_OF_BIN == p] > 0.5).mean(1) for p in range(7)]).mean(1)
    return any_ph, first_ph, ratio

F4 = {}
for (k, ext), E in EASH.items():
    for sd_name, sd in [("10 ms", 1), ("30 ms", 3), ("none", 0)]:
        F4[(k, ext, sd_name)] = fig4_measures(E, sd)

def chi2x2(a, na, b, nb, yates):
    t = np.array([[a, na - a], [b, nb - b]])
    return stats.chi2_contingency(t, correction=yates)[1]

def phase_test(phase, high, others):
    rows = []
    for (k, ext, sdn), (anyp, firstp, _) in F4.items():
        reg = REG[k]
        for rd, M in [("(a) any bin", anyp), ("(b) first crossing", firstp)]:
            prop = {r: M[reg == r, phase].mean() for r in S.REGIONS}
            cnt = {r: (M[reg == r, phase].sum(), (reg == r).sum()) for r in S.REGIONS}
            for yates in (False, True):
                row = {"set": k, "ext": ext, "smooth": sdn, "reading": rd, "Yates": yates}
                row.update({f"P {r}": round(prop[r], 3) for r in S.REGIONS})
                for o in others:
                    row[f"p vs {o}"] = chi2x2(*cnt[high], *cnt[o], yates)
                rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
r41 = phase_test(0, "VS", ["DLS", "DMS"])
r41["holds"] = (r41["P VS"] > .54) & (r41["P VS"] > r41["P DLS"]) & (r41["P VS"] > r41["P DMS"]) & (r41["p vs DLS"] < .05) & (r41["p vs DMS"] < .05)
print(r41.round(5).to_string())
ADOPT = (r41.set == "deposited PANs") & (~r41.ext) & (r41.smooth == "30 ms") & (r41.reading == "(a) any bin")
print("Adopted reading (deposited PANs, no extinction trials, 30 ms, any bin):")
print(r41[ADOPT].round(5).to_string())
print("By rat, proportion of deposited PANs activated in Phase 1 (adopted reading):")
_any = F4[("deposited PANs", False, "30 ms")][0]
print(pd.crosstab([n["subject"] for n in PAN], REG["deposited PANs"], values=_any[:, 0], aggfunc="mean").round(2).to_string())
main = r41[r41.set == "deposited PANs"]
print("Deposited PANs: holds in", int(main.holds.sum()), "of", len(main), "readings; Methods-rule set:",
      int(r41[r41.set != "deposited PANs"].holds.sum()), "of", int((r41.set != "deposited PANs").sum()))

**Result: reproduced.** Under the adopted reading (deposited PANs, extinction trials excluded, 30 ms smoothing, any bin at or above 0.5), 0.652 of VS PANs are activated in Phase 1, against 0.461 of DLS (chi-square p = 0.0005) and 0.339 of DMS (p < 1e-5). The claim holds in all 24 readings on the deposited PANs and in all 24 on the Methods-rule set.

In [ ]:
r45 = phase_test(4, "DMS", ["DLS", "VS"])
r45["holds"] = (r45["P DMS"] > .54) & (r45["P DMS"] > r45["P DLS"]) & (r45["P DMS"] > r45["P VS"]) & (r45["p vs DLS"] < .05) & (r45["p vs VS"] < .05)
print(r45.round(5).to_string())
print("Adopted reading:"); print(r45[ADOPT.to_numpy()].round(5).to_string())
main = r45[r45.set == "deposited PANs"]
print("Deposited PANs: holds in", int(main.holds.sum()), "of", len(main), "readings; Methods-rule set:",
      int(r45[r45.set != "deposited PANs"].holds.sum()), "of", int((r45.set != "deposited PANs").sum()))

**Result: reproduced, on the adopted reading.** With extinction trials excluded, 30 ms smoothing and the any-bin reading, 0.652 of DMS PANs are activated in Phase 5, against 0.436 in DLS (p = 0.00025) and 0.514 in VS (p = 0.029 without and 0.040 with Yates' correction). The DMS against VS difference is not significant with 10 ms or no smoothing (p 0.083 to 0.21) or with extinction trials included (p 0.050 to 0.067 at 30 ms), nor on the Methods-rule set (p 0.064 to 0.39). The first-crossing reading cannot give proportions above 0.6 in two phases (DMS 0.21 to 0.22) and fails.

In [ ]:
r47 = phase_test(6, "VS", ["DLS"])
r47["holds"] = (r47["P VS"] > r47["P DLS"]) & (r47["p vs DLS"] < .05)
print(r47.round(5).to_string())
print("Adopted reading:"); print(r47[ADOPT.to_numpy()].round(5).to_string())
main = r47[r47.set == "deposited PANs"]
print("Deposited PANs: holds in", int(main.holds.sum()), "of", len(main), "readings; Methods-rule set:",
      int(r47[r47.set != "deposited PANs"].holds.sum()), "of", int((r47.set != "deposited PANs").sum()))

**Result: reproduced, on the adopted reading.** Under the any-bin reading 0.659 of VS PANs are activated in Phase 7 against 0.520 of DLS (p = 0.010; 0.014 with Yates), and the any-bin reading holds in all 12 of its variants on each unit set. Under the first-crossing reading the proportions are 0.05 to 0.14 and do not differ (p 0.36 to 1).

In [ ]:
rows = []
for (k, ext, sdn), (_, _, ratio) in F4.items():
    reg = REG[k]
    m = {r: ratio[reg == r].mean() for r in S.REGIONS}
    p = {f"p {a}-{b}": stats.mannwhitneyu(ratio[reg == a], ratio[reg == b]).pvalue
         for a, b in [("VS", "DLS"), ("VS", "DMS"), ("DLS", "DMS")]}
    row = {"set": k, "ext": ext, "smooth": sdn}
    row.update({f"mean {r}": round(v, 4) for r, v in m.items()}); row.update(p)
    row["holds"] = m["VS"] > m["DMS"] > m["DLS"] and all(v < .05 for v in p.values())
    rows.append(row)
r4c = pd.DataFrame(rows)
print(r4c.round(5).to_string())
k0 = ("deposited PANs", False, "30 ms")
fig, ax = plt.subplots(1, 3, figsize=(10, 3))
for i, r in enumerate(S.REGIONS):
    E = EASH[("deposited PANs", False)][REG["deposited PANs"] == r]
    sm = np.array([S.gauss_smooth(e, 3) for e in E]); nrm = sm / sm.max(1, keepdims=True)
    order = np.argsort(np.argmax(nrm >= .5, axis=1))
    ax[i].imshow(nrm[order], aspect="auto", extent=[0, S.EDGES[-1], len(E), 0], vmin=0, vmax=1)
    for e in S.EDGES: ax[i].axvline(e, color="w", lw=.5)
    ax[i].set_title(f"{r} (n={len(E)})"); ax[i].set_xlabel("EASH time from E0 (s)")
plt.tight_layout(); plt.show()
print("Normalized EASHs, values 0 to 1 by construction; 30 ms smoothing, extinction trials excluded.")

**Result: partially reproduced.** Under the adopted reading (30 ms, extinction trials excluded) the mean activity ratio is 0.216 in VS, 0.130 in DMS and 0.117 in DLS: VS is above DLS and DMS (p < 1e-5), but DLS is not significantly below DMS (p = 0.065; 0.054 to 0.082 over all deposited readings). On the Methods-rule set DLS is below DMS with p 0.018 to 0.050 under 10 and 30 ms smoothing.

## Mutual information over all trials (Fig. 5 claims)

Per PAN, MI between the four firing levels and a binary event in each 100 ms bin of the EASH
axis (bins from E0; a bin belongs to the phase that holds its centre), plus the 100 ms
windows ending at E3 and at E4. Events: action (left or right), state (fixed-choice blocks,
single and double, against free-choice blocks), reward (rewarded or not). Trials: all choice
trials without (O3 a) or with (O3 b) extinction trials; with extinction trials, those of the
left and right tones count as fixed and those of the choice tone as free. O1 readings: (a)
plug-in MI; (b) the same minus the mean of the neuron's 100 shuffles; (c) plug-in MI minus
the Panzeri and Treves (1996) bias estimate. Reading (c) is added from the paper: the Results
cite Panzeri and Treves (1996) for the MI ("the amount of mutual information between neuronal
firing and each task event was calculated (Panzeri and Treves, 1996; Ito and Doya, 2009)"),
the paper that gives this correction. Threshold: second largest of 100 region averages with
permuted events, computed the same way. Mann-Whitney U tests two-sided.

In [ ]:
G100 = S.bins(0.1)
WIN = np.vstack([G100, [[S.EDGES[3] - .1, S.EDGES[3]], [S.EDGES[4] - .1, S.EDGES[4]]]])
WLEN = WIN[:, 1] - WIN[:, 0]
CEN = G100.mean(1)
PH100 = np.searchsorted(S.EDGES, CEN, "right") - 1  # phase index 0..6 of each grid bin
iE3, iE4 = len(G100), len(G100) + 1

def labels_for(n, var, ext):
    tr = n["tr"]; m = trial_mask(n, ext)
    if var == "action":
        y = (tr.action == 1).to_numpy()
    elif var == "reward":
        y = (tr.reward == 1).to_numpy()
    else:  # state: fixed (SF, DF, or fixed-tone extinction) = 1, free = 0
        y = (tr.cue != "choice").to_numpy()
    return m, y

def mi_set(nset, var, ext, seed):
    rng = np.random.default_rng(seed)
    out = {"plugin": ([], []), "pt": ([], [])}
    for n in nset:
        m, y = labels_for(n, var, ext)
        c = S.wcounts(n, WIN)[m]
        res = S.mi_with_shuffles(c, y[m], rng, 100, 1.0)
        for est, (r, sh) in res.items():
            out[est][0].append(r / WLEN); out[est][1].append(sh / WLEN)
    return {est: (np.array(a), np.array(b)) for est, (a, b) in out.items()}  # N x W, N x 100 x W

MI = {}
for k, v in SETS.items():
    for ext in (False, True):
        for j, var in enumerate(["action", "state", "reward"]):
            MI[(k, ext, var)] = mi_set(v, var, ext, SEED + 10 * j + ext)

def mi_reading(key, o1):
    """Per-neuron MI (N x W) and per-region threshold (W) under O1 reading a, b or c."""
    real, shs = MI[key]["pt" if o1 == "c" else "plugin"]
    if o1 == "b":
        mu = shs.mean(1)
        real, shs = real - mu, shs - mu[:, None, :]
    reg = REG[key[0]]
    avg = {r: real[reg == r].mean(0) for r in S.REGIONS}
    thr = {r: S.threshold(shs[reg == r].mean(0)) for r in S.REGIONS}
    return real, reg, avg, thr

def mwu(x, y):
    return stats.mannwhitneyu(x, y).pvalue

def readings():
    for k in SETS:
        for ext in (False, True):
            for o1 in ("a", "b", "c"):
                yield k, ext, o1

In [ ]:
# Clock check: action and reward information should rise only after the events that define them.
real, reg, avg, thr = mi_reading(("deposited PANs", False, "reward"), "a")
pre = PH100 <= 4; post = PH100 == 5
print("Reward MI (bits/s), all deposited PANs: mean over Phases 1-5", avg["VS"][:len(G100)][pre].mean().round(3),
      "; Phase 6", avg["VS"][:len(G100)][post].mean().round(3), "(VS)")
for r in S.REGIONS:
    a = avg[r][:len(G100)]; t = thr[r][:len(G100)]
    print(f"  {r}: bins above threshold in Phases 1-5: {int((a > t)[pre].sum())} of {int(pre.sum())}; in Phase 6: {int((a > t)[post].sum())} of {int(post.sum())}")
# shifted labels: reward of the next trial (true labels shifted by one trial) should carry little information after E5
rng = np.random.default_rng(SEED + 99)
sh_mi, tr_mi = [], []
for n in SETS["deposited PANs"]:
    m, y = labels_for(n, "reward", False)
    c = S.wcounts(n, WIN[(PH100 == 5).nonzero()[0]])[m]; yy = y[m]
    lev = S.quartile_levels(c)
    tr_mi.append(S.mi_bits(lev, yy).mean() / .1); sh_mi.append(S.mi_bits(lev, np.roll(yy, 1)).mean() / .1)
print("Phase 6 reward MI with true labels", np.mean(tr_mi).round(3), "against labels shifted by one trial", np.mean(sh_mi).round(3))

## fig5b-state-info-vs

> Information about the block type (fixed- or free-choice) is strongest in VS, especially
> during the approach to the center hole (Phase 1).

Readings (criterion): per-neuron MI (a) averaged over the Phase 1 bins; (b) in each Phase 1
bin, judged on the bins where the VS average passes its threshold (the claim must hold in
every such bin).

In [ ]:
rows = []
ph1 = np.where(PH100 == 0)[0]
for k, ext, o1 in readings():
    real, reg, avg, thr = mi_reading((k, ext, "state"), o1)
    vs_pass = [b for b in ph1 if avg["VS"][b] > thr["VS"][b]]
    x = real[:, ph1].mean(1)
    pa = {o: mwu(x[reg == "VS"], x[reg == o]) for o in ["DLS", "DMS"]}
    ma = {r: x[reg == r].mean() for r in S.REGIONS}
    ok_a = all(ma["VS"] > ma[o] and pa[o] < .05 for o in ["DLS", "DMS"]) and len(vs_pass) > 0
    okb = []
    for b in vs_pass:
        okb.append(all(real[reg == "VS", b].mean() > real[reg == o, b].mean() and mwu(real[reg == "VS", b], real[reg == o, b]) < .05 for o in ["DLS", "DMS"]))
    rows.append({"set": k, "ext": ext, "O1": o1, "mean DLS": ma["DLS"], "mean DMS": ma["DMS"], "mean VS": ma["VS"],
                 "p VS-DLS": pa["DLS"], "p VS-DMS": pa["DMS"], "VS bins above thr": f"{len(vs_pass)}/{len(ph1)}",
                 "(a) holds": ok_a, "(b) bins holding": f"{sum(okb)}/{len(okb)}", "(b) holds": len(okb) > 0 and all(okb)})
r5b = pd.DataFrame(rows)
print(r5b.round(4).to_string())
fig, ax = plt.subplots(1, 3, figsize=(11, 2.8), sharey=False)
for i, var in enumerate(["state", "action", "reward"]):
    real, reg, avg, thr = mi_reading(("deposited PANs", False, var), "a")
    for r in S.REGIONS:
        ax[i].plot(CEN, avg[r][:len(G100)], label=r); ax[i].plot(CEN, thr[r][:len(G100)], ls=":", lw=.8)
    for e in S.EDGES: ax[i].axvline(e, color="k", lw=.3)
    ax[i].set_title(f"{var} MI, deposited PANs"); ax[i].set_xlabel("EASH time (s)")
ax[0].set_ylabel("bits/s"); ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()
for var in ["state", "action", "reward"]:
    real, reg, avg, thr = mi_reading(("deposited PANs", False, var), "a")
    print(var, "region averages range", {r: (avg[r].min().round(3), avg[r].max().round(3)) for r in S.REGIONS})

**Result: partially reproduced, on the adopted reading.** With bias-corrected MI (Panzeri-Treves, reading c) averaged over Phase 1, VS carries more state information than DLS (0.053 against 0.013 bits/s, p < 1e-4) and more than DMS (0.023), but the VS against DMS test gives p = 0.071 (0.053 with extinction trials). The plug-in and shuffle-corrected readings give p = 0.022 to 0.036 for VS against DMS, and the Methods-rule set p = 0.017 to 0.035 under every reading. The VS average is above its threshold in all 20 Phase 1 bins. Per bin (reading b), VS exceeds both regions significantly in 8 to 13 of the 20 bins.

## fig5e-action-info-rises-dls

> Action information (left or right choice) starts rising during cue presentation (Phase 3)
> only in DLS.

In [ ]:
rows = []
ph3 = np.where(PH100 == 2)[0]
for k, ext, o1 in readings():
    real, reg, avg, thr = mi_reading((k, ext, "action"), o1)
    above = {r: [round(CEN[b], 2) for b in ph3 if avg[r][b] > thr[r][b]] for r in S.REGIONS}
    pre = np.where(PH100 <= 1)[0]
    rows.append({"set": k, "ext": ext, "O1": o1, **{f"{r} Phase 3 bins above": len(above[r]) for r in S.REGIONS},
                 **{f"{r} Phases 1-2 bins above": int((avg[r][pre] > thr[r][pre]).sum()) for r in S.REGIONS},
                 "DLS bin centres": above["DLS"][:6], "DMS bin centres": above["DMS"][:6], "VS bin centres": above["VS"][:6],
                 "holds": len(above["DLS"]) > 0 and not above["DMS"] and not above["VS"]})
r5r = pd.DataFrame(rows)
print(f"Phases 1 and 2 hold {int((PH100 <= 1).sum())} bins; Phase 3 holds {len(ph3)} bins, centres {CEN[ph3[0]]:.2f} to {CEN[ph3[-1]]:.2f} s (E2 = {S.EDGES[2]:.2f}, E3 = {S.EDGES[3]:.2f})")
print(r5r.to_string())
real, reg, avg, thr = mi_reading(("deposited PANs", False, "action"), "a")
print("Deposited PANs, O1 a: region average minus threshold in the Phase 3 bins (bits/s):")
print(pd.DataFrame({r: (avg[r] - thr[r])[ph3].round(4) for r in S.REGIONS}, index=CEN[ph3].round(2)).T.to_string())

**Result: not reproduced.** In every reading the action information of all three regions is above the shuffle threshold in all 15 Phase 3 bins (VS in 14 in four readings), and DLS is already above threshold in 12 to 17 of the 27 bins of Phases 1 and 2. A rise specific to DLS is visible in the margins above threshold: DLS grows from 0.003 to 0.060 bits/s across Phase 3, DMS stays between 0.003 and 0.015 and VS between 0.002 and 0.012. The threshold test of the criterion does not see it, as action information over all trials exceeds the random-event threshold before the cue.

## fig5e-action-info-cue-offset-dls

> In the 100 ms before cue offset, action information is significantly higher in DLS than in
> DMS and VS.

In [ ]:
def window_test(var, col, order_pairs):
    rows = []
    for k, ext, o1 in readings():
        real, reg, avg, thr = mi_reading((k, ext, var), o1)
        x = real[:, col] if np.ndim(col) == 0 else real[:, col].mean(1)
        row = {"set": k, "ext": ext, "O1": o1}
        row.update({f"mean {r}": x[reg == r].mean() for r in S.REGIONS})
        for a, b in order_pairs:
            row[f"p {a}-{b}"] = mwu(x[reg == a], x[reg == b])
        rows.append(row)
    return pd.DataFrame(rows)
r5c = window_test("action", iE3, [("DLS", "DMS"), ("DLS", "VS")])
r5c["holds"] = (r5c["mean DLS"] > r5c["mean DMS"]) & (r5c["mean DLS"] > r5c["mean VS"]) & (r5c["p DLS-DMS"] < .05) & (r5c["p DLS-VS"] < .05)
print(r5c.round(5).to_string())

**Result: reproduced, on the adopted reading.** In the 100 ms before cue offset, bias-corrected action MI (reading c) is 0.066 bits/s in DLS, 0.014 in DMS and 0.012 in VS; DLS against DMS p = 0.047 (0.039 with extinction trials) and DLS against VS p = 0.0009. With the plug-in or shuffle-corrected MI, DLS against DMS gives p = 0.060 to 0.21, and on the Methods-rule set p = 0.072 to 0.31, so the DMS part rests on the bias correction.

## fig5e-action-info-center-exit

> In the 100 ms before center-hole exit, action information in DMS is higher than in DLS (not
> significant) and VS, and is weakest in VS.

In [ ]:
r5x = window_test("action", iE4, [("DMS", "DLS"), ("DMS", "VS"), ("DLS", "VS")])
r5x["holds"] = (r5x["mean DMS"] > r5x["mean DLS"]) & (r5x["mean DLS"] > r5x["mean VS"]) & (r5x["p DMS-VS"] < .05) & (r5x["p DLS-VS"] < .05)
print(r5x.round(5).to_string())

**Result: reproduced.** In the 100 ms before centre-hole exit action MI is 0.216 to 0.230 bits/s in DMS, 0.113 to 0.122 in DLS and 0.015 to 0.025 in VS on the deposited PANs; DMS against VS and DLS against VS give p < 1e-4 and DMS against DLS p = 0.13 to 0.30, as in the paper (p = 0.16). All 12 readings hold.

## fig5d-action-info-phase5-dms

> During action execution (Phase 5), action information is highest in DMS.

Readings: (a) the mean over the Phase 5 bins; (b) the bin of the DMS region peak in Phase 5.

In [ ]:
ph5 = np.where(PH100 == 4)[0]
ra = window_test("action", ph5, [("DMS", "DLS"), ("DMS", "VS"), ("DLS", "VS")]); ra["reading"] = "(a) Phase 5 mean"
rows = []
for k, ext, o1 in readings():
    real, reg, avg, thr = mi_reading((k, ext, "action"), o1)
    b = ph5[np.argmax(avg["DMS"][ph5])]
    x = real[:, b]
    row = {"set": k, "ext": ext, "O1": o1, "reading": f"(b) DMS peak bin {CEN[b]:.2f} s"}
    row.update({f"mean {r}": x[reg == r].mean() for r in S.REGIONS})
    for a, c in [("DMS", "DLS"), ("DMS", "VS"), ("DLS", "VS")]:
        row[f"p {a}-{c}"] = mwu(x[reg == a], x[reg == c])
    rows.append(row)
r5d = pd.concat([ra, pd.DataFrame(rows)], ignore_index=True)
r5d["holds"] = (r5d["mean DMS"] > r5d["mean DLS"]) & (r5d["mean DMS"] > r5d["mean VS"]) & (r5d["p DMS-DLS"] < .05) & (r5d["p DMS-VS"] < .05)
print(r5d.round(5).to_string())

**Result: reproduced.** Over Phase 5 action MI averages 0.52 to 0.53 bits/s in DMS, 0.31 to 0.33 in DLS and 0.10 to 0.11 in VS; DMS against DLS p = 2e-5, DMS against VS and DLS against VS p < 1e-4. At the DMS peak bin (4.95 s) the same holds. All 24 readings hold.

## fig5g-reward-info

> Reward information rises in all regions after L/R-hole entry and is strongest in VS, then
> DMS, with DLS significantly below both.

Readings: per-neuron mean over the bins of (a) Phase 6, (b) Phases 6 and 7, (c) the bin of the
VS region peak. Part (1): the first bin at or after E5 where each region average passes its
threshold lies in Phase 6; the first crossing over the whole trial is shown too.

In [ ]:
rows = []
for k, ext, o1 in readings():
    real, reg, avg, thr = mi_reading((k, ext, "reward"), o1)
    first_after, first_any = {}, {}
    for r in S.REGIONS:
        ab = avg[r][:len(G100)] > thr[r][:len(G100)]
        idx = np.where(ab & (PH100 >= 5))[0]; first_after[r] = PH100[idx[0]] + 1 if len(idx) else None
        idx = np.where(ab)[0]; first_any[r] = PH100[idx[0]] + 1 if len(idx) else None
    c1 = all(first_after[r] == 6 for r in S.REGIONS)
    bpk = np.argmax(avg["VS"][:len(G100)])
    for rd, cols in [("(a) Phase 6", np.where(PH100 == 5)[0]), ("(b) Phases 6-7", np.where(PH100 >= 5)[0]), (f"(c) VS peak {CEN[bpk]:.2f} s", [bpk])]:
        x = real[:, cols].mean(1)
        m = {r: x[reg == r].mean() for r in S.REGIONS}
        p = {f"p {a}-{b}": mwu(x[reg == a], x[reg == b]) for a, b in [("DLS", "DMS"), ("DLS", "VS"), ("DMS", "VS")]}
        c2 = m["VS"] > m["DMS"] > m["DLS"] and p["p DLS-DMS"] < .05 and p["p DLS-VS"] < .05
        rows.append({"set": k, "ext": ext, "O1": o1, "reading": rd, **{f"mean {r}": m[r] for r in S.REGIONS}, **p,
                     "first phase after E5": [first_after[r] for r in S.REGIONS], "first phase overall": [first_any[r] for r in S.REGIONS],
                     "(1)": c1, "(2)": c2, "holds": c1 and c2})
r5g = pd.DataFrame(rows)
print(r5g.round(5).to_string())

**Result: blocked (methods_underspecified).** Part (1) holds in every reading: each region's reward information first passes its threshold after E5 in Phase 6. Part (2) depends on the window, which the paper does not give. At the bin of the VS peak (7.05 s, in Phase 7) the means order VS 0.34 > DMS 0.27 > DLS 0.22 with DLS below DMS (p = 0.031 to 0.049) and VS (p < 0.001) on the deposited PANs. Over Phase 6, DMS (0.077) is above VS (0.075) and DLS against DMS gives p = 0.36 to 0.47. Over Phases 6 and 7, DLS against VS gives p = 0.12 to 0.16. On the Methods-rule set no window satisfies part (2).

## Fig. 6 claims (fig6-event-coding-proportions, fig6-above-chance, fig6bc-similar-fixed-free)

A PAN codes an event when its spike counts in the event's window differ between the two
conditions by a two-sided Mann-Whitney U test with p < 0.01 (Fig. 6 legend). Windows (Fig. 6
legend): AC the 500 ms before E4, action the 500 ms after E4, Reward1 the 500 ms before E6,
Reward2 the 500 ms after E6; state (fixed against free blocks, all trials only) (a) all of
Phase 3, (b) the 500 ms before E3. The legend does not say whether the windows are on the
EASH axis or in real time around the events; both are run. Trial sets: fixed-choice trials
(Fig. 6B), free-choice trials (6C), all trials (6D; O3 without or with extinction trials).

In [ ]:
EV_WIN = {"AC": (4, -.5, 0), "action": (4, 0, .5), "Reward1": (6, -.5, 0), "Reward2": (6, 0, .5),
          "state (a)": None, "state (b)": (3, -.5, 0)}
def event_counts(n, ev, axis):
    if ev == "state (a)":
        return S.wcounts(n, [[S.EDGES[2], S.EDGES[3]]])[:, 0]
    e, a, b = EV_WIN[ev]
    if axis == "EASH":
        return S.wcounts(n, [[S.EDGES[e] + a, S.EDGES[e] + b]])[:, 0]
    return S.rcounts(n, e, [[a, b]])[:, 0]

def coding_table(nset, axis):
    rows = []
    for n in nset:
        tr = n["tr"]
        sets = {"fixed": tr.btype.isin(["SF", "DF"]).to_numpy(), "free": (tr.btype == "FREE").to_numpy(),
                "all": (tr.btype != "EXT").to_numpy(), "all+ext": np.ones(len(tr), bool)}
        for ev in EV_WIN:
            c = event_counts(n, ev, axis)
            for sname, m in sets.items():
                if ev.startswith("state"):
                    if not sname.startswith("all"):
                        continue
                    y = (tr.cue != "choice").to_numpy()
                elif ev in ("AC", "action"):
                    y = (tr.action == 1).to_numpy()
                else:
                    y = (tr.reward == 1).to_numpy()
                x1, x0 = c[m & y], c[m & ~y]
                p = stats.mannwhitneyu(x1, x0).pvalue if len(x1) and len(x0) and np.ptp(np.r_[x1, x0]) > 0 else 1.0
                rows.append({"region": n["region"], "subject": n["subject"], "event": ev, "trials": sname, "p": p})
    return pd.DataFrame(rows)
CODING = {(k, ax): coding_table(v, ax) for k, v in SETS.items() for ax in ("EASH", "real")}

def props(k, ax):
    ct = CODING[(k, ax)]
    ct = ct.assign(coding=ct.p < .01)
    return ct.groupby(["trials", "event", "region"]).coding.agg(["sum", "size", "mean"]).reset_index()

In [ ]:
rows = []
for k in SETS:
    for ax in ("EASH", "real"):
        P = props(k, ax)
        for ts in ("all", "all+ext"):
            Pa = P[P.trials == ts].pivot(index="event", columns="region", values="mean")
            Ps = P[P.trials == ts].pivot(index="event", columns="region", values="sum")
            Pn = P[P.trials == ts].pivot(index="event", columns="region", values="size")
            for st in ("state (a)", "state (b)"):
                want = {st: "VS", "Reward1": "VS", "AC": "DLS", "action": "DMS"}
                got = {e: Pa.loc[e].idxmax() for e in want}
                row = {"set": k, "axis": ax, "trials": ts, "state window": st}
                row.update({f"{e} largest": got[e] for e in want if e != st}); row["state largest"] = got[st]
                row["holds"] = all(got[e] == want[e] for e in want)
                rows.append(row)
            if k == "deposited PANs" and ts == "all":
                print(f"Deposited PANs, {ax} axis, all trials without extinction: proportion coding (count/n)")
                print((Pa.round(3).astype(str) + " (" + Ps.astype(int).astype(str) + "/" + Pn.astype(int).astype(str) + ")").to_string())
                for e in Pa.index:
                    ps = {f"{a}-{b}": stats.chi2_contingency([[Ps.loc[e, a], Pn.loc[e, a] - Ps.loc[e, a]], [Ps.loc[e, b], Pn.loc[e, b] - Ps.loc[e, b]]], correction=False)[1]
                          for a, b in [("DLS", "DMS"), ("DLS", "VS"), ("DMS", "VS")]}
                    print(f"   chi2 (no Yates) {e}: " + ", ".join(f"{q} p={v:.4f}" for q, v in ps.items()))
r6 = pd.DataFrame(rows)
print(r6.to_string())
P = props("deposited PANs", "EASH")
ct = CODING[("deposited PANs", "EASH")]
print("\nBy rat (deposited PANs, EASH axis, all trials), proportion coding per event:")
print(ct[ct.trials == "all"].assign(c=lambda d: d.p < .01).groupby(["event", "subject"]).c.mean().unstack().round(2).to_string())

**Result: partially reproduced, on the adopted trials.** On the deposited PANs, with extinction trials excluded and windows on the EASH axis, VS has the largest Reward1 proportion (0.435 against 0.319 and 0.321), DLS the largest AC proportion (0.422 against 0.411 in DMS, chi-square p = 0.85) and DMS the largest action proportion (0.839). The state part depends on the state window: with the 500 ms before E3, VS is largest (0.362 against 0.348 and 0.289); with all of Phase 3, DMS is largest (0.509 against 0.493 in VS). No pair differs significantly for state (p 0.15 to 0.80). With extinction trials included, DMS overtakes DLS for AC on the EASH axis. On the Methods-rule set all four parts hold in every reading.

In [ ]:
rows = []
for k in SETS:
    for ax in ("EASH", "real"):
        P = props(k, ax)
        P["binom p"] = [stats.binomtest(int(s), int(n), .01, alternative="greater").pvalue for s, n in zip(P["sum"], P["size"])]
        P["set"] = k; P["axis"] = ax
        rows.append(P)
r6c = pd.concat(rows, ignore_index=True)
r6c["sig"] = r6c["binom p"] < .01
print(r6c.groupby(["set", "axis", "trials"]).sig.agg(["sum", "size"]))
print("Tests not significant:")
print(r6c[~r6c.sig].to_string())
print("Smallest proportion:", r6c.loc[r6c["mean"].idxmin(), ["set", "axis", "trials", "event", "region", "sum", "size", "mean", "binom p"]].to_dict())
print("All tests p < 0.01:", bool(r6c.sig.all()), "; deposited PANs:", bool(r6c[r6c.set == "deposited PANs"].sig.all()))

**Result: reproduced.** Every proportion of event-coding neurons, in every region, event and trial set, both unit sets and both window axes, is above 0.01 by a one-sided binomial test at p < 0.01 (the smallest proportion is 0.094, 13 of 138 VS PANs for AC in free-choice trials with real-time windows, binomial p = 1.8e-9).

In [ ]:
rows = []
for k in SETS:
    for ax in ("EASH", "real"):
        P = props(k, ax)
        for ev in ["AC", "action", "Reward1", "Reward2"]:
            f = P[(P.trials == "fixed") & (P.event == ev)].set_index("region")
            g = P[(P.trials == "free") & (P.event == ev)].set_index("region")
            row = {"set": k, "axis": ax, "event": ev, "largest fixed": f["mean"].idxmax(), "largest free": g["mean"].idxmax()}
            for r in S.REGIONS:
                pr = stats.chi2_contingency([[f.loc[r, "sum"], f.loc[r, "size"] - f.loc[r, "sum"]], [g.loc[r, "sum"], g.loc[r, "size"] - g.loc[r, "sum"]]], correction=False)[1]
                row[r] = f"{f.loc[r, 'mean']:.2f}/{g.loc[r, 'mean']:.2f} p={pr:.3f}"
            row["same"] = row["largest fixed"] == row["largest free"]
            rows.append(row)
r6b = pd.DataFrame(rows)
print("Region columns: proportion fixed/free and chi-square p of the difference")
print(r6b.to_string())

**Result: partially reproduced.** On the deposited PANs the region with the largest proportion is the same in fixed- and free-choice trials for action (DMS), Reward1 (VS) and Reward2 (VS), on both axes, but not for AC (fixed DMS 0.38 against DLS 0.36; free DLS 0.31 against DMS 0.24). Of the 12 region and event differences between fixed and free trials (EASH axis), one is significant (DMS AC, 0.38 against 0.24, p = 0.021). On the Methods-rule set the largest region agrees for all four events.

## Regression with BIC selection (Fig. 8 claims and no-policy-coding)

QL(t) and QR(t) are the predictive means of the FQ model with time-varying parameters, run
over each session's free-choice sequence with the drift SDs chosen on the training half of
the alternate split (the fit above). Regressors (Results, model 2): constant, trial index t,
a(t) (1 left, 2 right), r(t), a(t-1), r(t-1) (the previous choice trial of the session), QL(t),
QR(t). For each neuron and window, the subset with the smallest BIC is chosen among all
subsets of the seven non-constant terms with the constant always in (claims file O7, reading
c), and a term is coded when its coefficient has p < 0.01 in that model (Methods). Readings:
trials (O7 a) free-choice trials only, (O7 b) all choice trials outside extinction blocks with
QL and QR carried through the session; windows (O6) 500 ms before and after (i) E1 to E7 or
(ii) E0 to E6, (a) in real time or (b) on the EASH axis (EASH windows beyond E0 or E7 do not
exist and are dropped). A window belongs to the phase that holds its centre on the EASH axis.
Proportions are tested against a chance level of 0.01 by a one-sided binomial test at p <
0.01 (Fig. 8 legend; chance as the claims file reads it). Mann-Whitney U tests on 0/1
indicators (O5).

In [ ]:
sa, sk = MODEL_RES["alt"][1]["FQ variable"]
rngq = np.random.default_rng(SEED + 30)
QV = {}
for s in names:
    a, r = SESS[s]
    _, mq = M.pf_session("FQ", a, r, sa, sk, rngq, return_q=True)
    g = fc[fc.session == s]
    QV[s] = (g.trial.to_numpy(), mq)
print("FQ variable drift SDs used:", sa, sk)

# value after the update of a free-choice trial: the predictive mean for the next free trial
def q_carried(ses, trials, is_free):
    ft, mq = QV.get(ses, (np.array([], int), np.zeros((0, 2))))
    out = np.zeros((len(trials), 2))
    for i, (t, f) in enumerate(zip(trials, is_free)):
        j = np.searchsorted(ft, t, "left")
        if f:
            out[i] = mq[j]
        elif j < len(ft):
            out[i] = mq[j]  # predictive mean for the next free trial = value after the last one
        elif len(ft):
            out[i] = mq[-1]
    return out

NAMES = ["const", "t", "a", "r", "a1", "r1", "QL", "QR"]
def win_list(evset, axis):
    evs = range(1, 8) if evset == "i" else range(0, 7)
    out = []
    for e in evs:
        for side, (a, b) in (("before", (-.5, 0)), ("after", (0, .5))):
            lo, hi = S.EDGES[e] + a, S.EDGES[e] + b
            if axis == "EASH" and (lo < -1e-9 or hi > S.EDGES[-1] + 1e-9):
                continue
            ph = int(np.searchsorted(S.EDGES, (lo + hi) / 2, "right"))  # 1..7, 0 before E0, 8 after E7
            out.append((f"{side} E{e}", e, a, b, ph))
    return out

def regress_all(nset, trialset, evset, axis):
    wl = win_list(evset, axis)
    rows = []
    for n in nset:
        tr = n["tr"]
        allch = T[(T.session == n["session"]) & T.choice].sort_values("trial")
        prev = {t: (a_, r_) for t, a_, r_ in zip(allch.trial.to_numpy()[1:], allch.action.to_numpy()[:-1], allch.reward.to_numpy()[:-1])}
        m = ((tr.btype == "FREE") if trialset == "a" else (tr.btype != "EXT")).to_numpy().copy()
        m &= np.array([t in prev for t in tr.trial])
        idx = np.where(m)[0]
        if len(idx) < 20:
            continue
        tt = tr.iloc[idx]
        q = q_carried(n["session"], tt.trial.to_numpy(), (tt.btype == "FREE").to_numpy())
        pa = np.array([prev[t] for t in tt.trial])
        X = np.column_stack([np.ones(len(idx)), tt.trial.to_numpy(), tt.action.to_numpy(), tt.reward.to_numpy(),
                             pa[:, 0], pa[:, 1], q[:, 0], q[:, 1]]).astype(float)
        if axis == "EASH":
            Y = S.wcounts(n, [[S.EDGES[e] + a, S.EDGES[e] + b] for _, e, a, b, _ in wl], idx)
        else:
            Y = np.column_stack([S.rcounts(n, e, [[a, b]], idx)[:, 0] for _, e, a, b, _ in wl])
        sel = M.bic_select(X, Y.astype(float), NAMES)
        for (wn, e, a, b, ph), res in zip(wl, sel):
            row = {"region": n["region"], "subject": n["subject"], "unit": n["unit"], "session": n["session"], "window": wn, "phase": ph, "n": len(idx)}
            for v in NAMES[1:]:
                row[v] = v in res and res[v][1] < .01
                row[v + "_b"] = res[v][0] if v in res else 0.0
            rows.append(row)
    df = pd.DataFrame(rows)
    df["AV"] = df.QL | df.QR
    df["SV"] = df.QL & df.QR & (np.sign(df.QL_b) == np.sign(df.QR_b))
    df["POL"] = df.QL & df.QR & (np.sign(df.QL_b) != np.sign(df.QR_b))
    return df

REGR = {}
for trialset in ("a", "b"):
    for evset in ("i", "ii"):
        for axis in ("real", "EASH"):
            REGR[(trialset, evset, axis)] = regress_all(SETS["deposited PANs"], trialset, evset, axis)
print({k: len(v) for k, v in REGR.items()})

def prop_table(df, var):
    g = df.groupby(["window", "phase", "region"])[var].agg(["sum", "size"]).reset_index()
    g["prop"] = g["sum"] / g["size"]
    g["binom p"] = [stats.binomtest(int(s), int(n), .01, alternative="greater").pvalue for s, n in zip(g["sum"], g["size"])]
    return g

In [ ]:
rows = []
for key, df in REGR.items():
    P = {v: prop_table(df, v) for v in ("a", "r", "r1", "a1")}
    sig = {v: P[v][P[v]["binom p"] < .01] for v in P}
    e4 = {"before": ["before E4"], "after": ["after E4"]}
    c1 = all(((sig["a"].region == reg) & sig["a"].window.isin(["before E4", "before E3", "after E3", "before E2", "after E2"]) ).any()
             and ((sig["a"].region == reg) & sig["a"].window.isin(["after E4", "before E5", "after E5", "before E6"])).any() for reg in S.REGIONS)
    after_e5 = [w for w in df.window.unique() if w in ("after E5", "before E6", "after E6", "before E7", "after E7")]
    c2 = all(((sig["r"].region == reg) & sig["r"].window.isin(after_e5)).any() for reg in S.REGIONS)
    phases = sorted(p for p in df.phase.unique() if 1 <= p <= 7)
    miss3 = [(reg, ph) for reg in S.REGIONS for ph in phases if not ((sig["r1"].region == reg) & (sig["r1"].phase == ph)).any()]
    c3 = not miss3
    c4 = ((sig["a1"].region == "DMS") & (sig["a1"].phase == 5)).any()
    rows.append({"trials": key[0], "events": key[1], "axis": key[2], "(1) a(t) before and after E4": c1, "(2) r(t) after E5": c2,
                 "(3) r(t-1) every phase": c3, "(3) region, phase without": miss3, "(4) a(t-1) DMS Phase 5": c4, "holds": c1 and c2 and c3 and c4})
r8 = pd.DataFrame(rows)
print(r8.to_string())
key0 = ("a", "i", "real")
for kk in [("a", "ii", "EASH")]:
    P = prop_table(REGR[kk], "r1")
    print("r(t-1), free-choice trials, E0-E6, EASH axis, Phase 7 windows:")
    print(P[P.phase == 7].round(3).to_string())
for v in ("a", "a1", "r", "r1"):
    P = prop_table(REGR[key0], v)
    print(f"\n{v}: proportion coding, free-choice trials, E1-E7, real time (* binomial p < 0.01)")
    P["cell"] = P.prop.round(2).astype(str) + np.where(P["binom p"] < .01, "*", "")
    print(P.pivot(index=["phase", "window"], columns="region", values="cell").to_string())

**Result: partially reproduced, on the adopted readings.** On free-choice trials, a(t) is coded above chance in every region before and after E4, r(t) after E5, and a(t-1) in DMS in Phase 5, under every reading. r(t-1) is coded above chance in every region and phase with the windows around E1 to E7, but with the windows around E0 to E6 DMS has no window above chance in Phase 7, whose only window is then the one after E6 (0.036, binomial p = 0.027). With all choice trials (O7 b) all four parts hold under both event sets.

In [ ]:
def ph5_windows(df):
    return sorted(w for w in df.window.unique() if (df[df.window == w].phase == 5).all())
rows = []
for key, df in REGR.items():
    P = prop_table(df, "AV")
    for w in ph5_windows(df):
        d = df[df.window == w]
        pr = {r: d[d.region == r].AV.mean() for r in S.REGIONS}
        pl = stats.mannwhitneyu(d[d.region == "DMS"].AV.astype(float), d[d.region == "DLS"].AV.astype(float)).pvalue
        pv = stats.mannwhitneyu(d[d.region == "DMS"].AV.astype(float), d[d.region == "VS"].AV.astype(float)).pvalue
        rows.append({"trials": key[0], "events": key[1], "axis": key[2], "window": w, **{f"AV {r}": round(pr[r], 3) for r in S.REGIONS},
                     "p DMS-DLS": pl, "p DMS-VS": pv, "holds": max(pr, key=pr.get) == "DMS" and pl < .05})
r8e = pd.DataFrame(rows)
print(r8e.round(4).to_string())
print(r8e.groupby(["trials", "events", "axis"]).holds.any().rename("holds in a Phase 5 window").to_string())
d = REGR[("a", "i", "EASH")]; d = d[d.window == "before E5"]
print("\nBy rat, action-value coding before E5 (free-choice trials, EASH axis):")
print(d.groupby(["subject", "region"]).AV.agg(["mean", "size"]).unstack().round(2).to_string())
P = prop_table(REGR[key0], "AV"); P["cell"] = P.prop.round(2).astype(str) + np.where(P["binom p"] < .01, "*", "")
print("\nAction-value coding, free-choice trials, E1-E7, real time:")
print(P.pivot(index=["phase", "window"], columns="region", values="cell").to_string())

**Result: reproduced, on the adopted readings.** On free-choice trials with windows on the EASH axis, the proportion of action-value-coding neurons in the 500 ms before E5 is 0.321 in DMS, 0.261 in VS and 0.176 in DLS (DMS against DLS p = 0.0034, against VS p = 0.29). With real-time windows VS is highest (0.341, DMS 0.295), and with all choice trials VS is highest in every Phase 5 window, so the claim rests on the EASH-axis windows and the free-choice trials.

In [ ]:
rows = []
for key, df in REGR.items():
    for w in ph5_windows(df):
        d = df[df.window == w]
        dm = d[d.region == "DMS"]
        frac = dm[dm.AV].QR.mean() if dm.AV.any() else np.nan
        p = stats.mannwhitneyu(dm.QR.astype(float), d[d.region == "DLS"].QR.astype(float)).pvalue
        rows.append({"trials": key[0], "events": key[1], "axis": key[2], "window": w, "DMS AV neurons": int(dm.AV.sum()),
                     "of them coding QR": frac, "QR DMS": dm.QR.mean(), "QR DLS": d[d.region == "DLS"].QR.mean(), "p": p,
                     "holds": frac > .5 and dm.QR.mean() > d[d.region == "DLS"].QR.mean() and p < .05})
r8q = pd.DataFrame(rows)
print(r8q.round(4).to_string())

**Result: not reproduced, on the adopted readings.** On free-choice trials with EASH-axis windows, 0.60 (after E4) and 0.47 (before E5) of the DMS action-value neurons code QR, and the QR proportion in DMS (0.16, 0.15) is not significantly above DLS (0.11; p = 0.18 and 0.32). With real-time windows the claim holds (0.64 to 0.70 code QR; DMS 0.17 to 0.19 against DLS 0.07 to 0.08, p = 0.004 to 0.009).

In [ ]:
rows = []
for key, df in REGR.items():
    phases = sorted(p for p in df.phase.unique() if 1 <= p <= 7)
    largest = {}
    for ph in phases:
        ok = False
        for w in df[df.phase == ph].window.unique():
            d = df[df.window == w]
            pr = {r: d[d.region == r].SV.mean() for r in S.REGIONS}
            ok |= pr["VS"] > max(pr["DLS"], pr["DMS"])
        largest[ph] = ok
    def vs_test(w):
        d = df[df.window == w]
        if not len(d):
            return False, np.nan, np.nan
        x = d[d.region == "VS"].SV.astype(float)
        p1 = stats.mannwhitneyu(x, d[d.region == "DLS"].SV.astype(float)).pvalue
        p2 = stats.mannwhitneyu(x, d[d.region == "DMS"].SV.astype(float)).pvalue
        ok = x.mean() > d[d.region == "DLS"].SV.mean() and x.mean() > d[d.region == "DMS"].SV.mean() and p1 < .05 and p2 < .05
        return ok, p1, p2
    ph1 = [vs_test(w) + (w,) for w in df[df.phase == 1].window.unique()]
    ph4 = vs_test("after E3")
    rows.append({"trials": key[0], "events": key[1], "axis": key[2], "VS largest in phases": [ph for ph in phases if largest[ph]],
                 "Phase 1 tests (ok, p DLS, p DMS, window)": [(o, round(a, 4), round(b, 4), w) for o, a, b, w in ph1],
                 "first half Phase 4 (after E3)": (ph4[0], round(ph4[1], 4), round(ph4[2], 4)),
                 "holds": all(largest.values()) and any(o for o, *_ in ph1) and ph4[0]})
r8f = pd.DataFrame(rows)
print(r8f.to_string())
P = prop_table(REGR[key0], "SV"); P["cell"] = P.prop.round(3).astype(str) + np.where(P["binom p"] < .01, "*", "")
print("\nState-value coding, free-choice trials, E1-E7, real time:")
print(P.pivot(index=["phase", "window"], columns="region", values="cell").to_string())

**Result: partially reproduced, on the adopted readings.** VS has the largest proportion of state-value neurons in every phase, and in Phase 1 (500 ms before E1) VS exceeds DLS (p = 0.0002) and DMS (p = 0.029). In the window after E3 (first half of Phase 4) VS exceeds DLS (p = 0.0005) but not DMS (p = 0.76). With all choice trials the Phase 1 test against DMS also fails (p = 0.10).

In [ ]:
rows = []
for key, df in REGR.items():
    P = prop_table(df, "POL")
    rows.append({"trials": key[0], "events": key[1], "axis": key[2], "largest proportion": P.prop.max(),
                 "where": tuple(P.loc[P.prop.idxmax(), ["region", "window"]]), "smallest binomial p": P["binom p"].min(),
                 "region-windows with p < 0.01": int((P["binom p"] < .01).sum())})
rp = pd.DataFrame(rows)
print(rp.round(5).to_string())

**Result: reproduced.** Under every reading no region and window has a proportion of policy-coding neurons above chance: the largest proportion is 0.015 to 0.036 and the smallest binomial p 0.027.

## Action information by block type (Fig. 9 and Fig. 10 claims)

Per PAN, action MI from the last 20 choice trials of each block (Fig. 9 legend and Results),
in a sliding window of the preceding 500 ms with 50 ms steps on the EASH axis (Fig. 9
legend), plus the 500 ms windows ending at E2, E3 and E4. Block types: single fixed (SF),
double fixed (DF), free with higher probabilities (90, 50) or (50, 90) (FH), free with lower
probabilities (50, 10) or (10, 50) (FL). Readings of how the blocks of one type are combined:
(a) their last 20 trials pooled into one MI estimate per neuron; (b) MI per block, averaged
over the blocks. Blocks with fewer than 20 choice trials give all of theirs. O1 readings a, b, c as before.

In [ ]:
T_END = np.round(np.arange(0.5, S.EDGES[-1] + 1e-9, 0.05), 3)
SLW = np.column_stack([T_END - .5, T_END])
SPW = np.array([[S.EDGES[e] - .5, S.EDGES[e]] for e in (2, 3, 4)])  # ending at E2, E3, E4
W9 = np.vstack([SLW, SPW])
iw = {"E2": len(SLW), "E3": len(SLW) + 1, "E4": len(SLW) + 2}
PH_END = np.searchsorted(S.EDGES, T_END, "left")  # phase (1..7) containing each window end (end on an edge counts to the earlier phase)

def btype9(tr):
    bt = tr.btype.to_numpy().astype(object)
    hi = (tr.btype == "FREE") & ((tr.rule_L == 90) | (tr.rule_R == 90))
    lo = (tr.btype == "FREE") & ((tr.rule_L == 10) | (tr.rule_R == 10))
    bt[hi.to_numpy()] = "FH"; bt[lo.to_numpy()] = "FL"
    return bt

def blocks_last20(n):
    tr = n["tr"]; bt = btype9(tr)
    out = {}
    for b in np.unique(tr.block):
        idx = np.where(tr.block.to_numpy() == b)[0]
        typ = bt[idx[0]]
        if typ in ("SF", "DF", "FH", "FL"):
            out.setdefault(typ, []).append(idx[-20:])
    out["FIX"] = out.get("SF", []) + out.get("DF", [])
    out["FREE"] = out.get("FH", []) + out.get("FL", [])
    return out

def mi9(n, rng):
    """Action MI (bits/s) per block type, readings a (pooled) and b (per block), with shuffles."""
    tr = n["tr"]; y = (tr.action == 1).to_numpy()
    C = S.wcounts(n, W9)
    res = {}
    for typ, blks in blocks_last20(n).items():
        if not blks:
            continue
        idx = np.concatenate(blks)
        pooled = S.mi_with_shuffles(C[idx], y[idx], rng, 100, 0.5)
        per = [S.mi_with_shuffles(C[bi], y[bi], rng, 100, 0.5) for bi in blks]
        for est in ("plugin", "pt"):
            res[(typ, "a", est)] = pooled[est]
            res[(typ, "b", est)] = (np.mean([p[est][0] for p in per], 0), np.mean([p[est][1] for p in per], 0).astype(np.float32))
    return res

R9 = {}
for k, v in SETS.items():
    rng = np.random.default_rng(SEED + 900 + (k != "deposited PANs"))
    R9[k] = [mi9(n, rng) for n in v]

def get9(k, typ, comb, o1):
    """Per-neuron MI (N x W), regions, region averages and thresholds, for neurons that have the block type."""
    est = "pt" if o1 == "c" else "plugin"
    have = [i for i, r in enumerate(R9[k]) if (typ, comb, est) in r]
    real = np.array([R9[k][i][(typ, comb, est)][0] for i in have])
    shs = np.array([R9[k][i][(typ, comb, est)][1] for i in have])
    if o1 == "b":
        mu = shs.mean(1); real = real - mu; shs = shs - mu[:, None, :]
    reg = REG[k][have]
    avg = {r: real[reg == r].mean(0) for r in S.REGIONS}
    thr = {r: S.threshold(shs[reg == r].mean(0)) for r in S.REGIONS}
    return real, reg, avg, thr, np.array(have)

def rise(avg, thr):
    """First crossing overall, and start of the sustained rise (above threshold from there through Phase 5)."""
    a = avg[:len(SLW)] > thr[:len(SLW)]
    first = np.argmax(a) if a.any() else None
    end5 = np.where(PH_END <= 5)[0][-1]
    sus = None
    for i in range(end5, -1, -1):
        if a[i]:
            sus = i
        else:
            break
    return first, sus

def readings9():
    for k in SETS:
        for comb in ("a", "b"):
            for o1 in ("a", "b", "c"):
                yield k, comb, o1

In [ ]:
rows = []
for k, comb, o1 in readings9():
    row = {"set": k, "comb": comb, "O1": o1}
    ok1, ok2 = True, True
    for typ in ("SF", "DF", "FH", "FL"):
        real, reg, avg, thr, _ = get9(k, typ, comb, o1)
        t_s = {}
        for r in S.REGIONS:
            f, s = rise(avg[r], thr[r])
            t_s[r] = s
            row[f"{typ} {r}"] = (f"first {T_END[f]:.2f} (P{PH_END[f]}); " if f is not None else "none; ") + \
                                (f"rise {T_END[s]:.2f} (P{PH_END[s]})" if s is not None else "no rise")
        ph = {r: PH_END[t_s[r]] if t_s[r] is not None else None for r in S.REGIONS}
        if typ in ("SF", "DF"):
            ok1 &= ph["DLS"] == 3 and ph["DMS"] == 4 and ph["VS"] == 5
        else:
            ok2 &= None not in t_s.values() and t_s["DLS"] < t_s["DMS"] < t_s["VS"]
    row["(1)"] = ok1; row["(2)"] = ok2; row["holds"] = ok1 and ok2
    rows.append(row)
r9 = pd.DataFrame(rows)
print("Phases: E1", S.EDGES[1], "E2", S.EDGES[2], "E3", S.EDGES[3], "E4", round(S.EDGES[4], 2), "E5", round(S.EDGES[5], 2), "E6", round(S.EDGES[6], 2))
print(r9.T.to_string())
print("Neurons with each block type (deposited):", {t: len(get9("deposited PANs", t, "a", "a")[4]) for t in ("SF", "DF", "FH", "FL")})
fig, ax = plt.subplots(1, 4, figsize=(13, 2.8), sharey=True)
for i, typ in enumerate(("SF", "DF", "FH", "FL")):
    real, reg, avg, thr, _ = get9("deposited PANs", typ, "a", "a")
    for r in S.REGIONS:
        ax[i].plot(T_END, avg[r][:len(SLW)], label=r); ax[i].plot(T_END, thr[r][:len(SLW)], ls=":", lw=.8)
    for e in S.EDGES: ax[i].axvline(e, color="k", lw=.3)
    ax[i].set_title(f"{typ} (pooled, plug-in)"); ax[i].set_xlabel("window end (s)")
ax[0].set_ylabel("action MI (bits/s)"); ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

**Result: not reproduced.** No reading gives the rise of DLS in Phase 3, DMS in Phase 4 and VS in Phase 5 in both fixed-choice block types. With the last 20 trials pooled, all three regions start their sustained rise between 2.75 and 3.20 s, at cue onset or early in Phase 3, in single and double fixed-choice blocks. With MI per block, the single fixed-choice blocks on the deposited PANs show the order (DLS 4.10 s in Phase 3, DMS 4.40 to 4.50 in Phase 4, VS 5.05 to 5.10 in Phase 5), but in double fixed-choice blocks all three rise in Phase 3 (3.20 to 3.95 s). In free-choice blocks the order DLS, DMS, VS holds in 6 of the 12 readings. In most conditions the region averages pass the threshold already in Phase 1 (first crossings 0.50 to 1.20 s).

In [ ]:
rows = []
for k, comb, o1 in readings9():
    for typ in ("SF", "DF", "FH", "FL"):
        real, reg, avg, thr, _ = get9(k, typ, comb, o1)
        for w in ("E3", "E4"):
            x = real[:, iw[w]]
            md, ml = x[reg == "DMS"].mean(), x[reg == "DLS"].mean()
            p = mwu(x[reg == "DMS"], x[reg == "DLS"])
            rows.append({"set": k, "comb": comb, "O1": o1, "block": typ, "window end": w, "mean DMS": md, "mean DLS": ml,
                         "p": p, "DMS sig. above DLS": md > ml and p < .05})
r9d = pd.DataFrame(rows)
print(r9d.round(4).to_string())
print("Cases with DMS significantly above DLS:", int(r9d["DMS sig. above DLS"].sum()), "of", len(r9d))

**Result: reproduced, on the adopted reading.** With bias-corrected MI (reading c), DMS is not significantly above DLS in any block type or window, pooled or per block, on either unit set. Of all 96 tests one shows DMS above DLS (per block, shuffle-corrected, single fixed-choice, window ending at E4: 0.039 against 0.018 bits/s, p = 0.021).

## fig10a-dls-fixed-above-free, fig10b-dms-fixed-above-free, fig10c-vs-no-fixed-free-difference

Windows: the 500 ms ending at E3 (end of Phase 3) and at E4 (end of Phase 4); the 500 ms
ending at E2 is shown as a variant for "500 ms before cue presentation". Fixed is SF and DF,
free is FH and FL, combined (a) as the per-neuron mean of the two block types or (b) with the
trials pooled (for block-combination reading b, pooled means all SF and DF blocks averaged).
Test: (a) Wilcoxon signed-rank over neurons (paired), (b) Mann-Whitney U (unpaired).

In [ ]:
def fixed_free(k, comb, o1, region, how):
    if how == "mean":
        A = [get9(k, t, comb, o1) for t in ("SF", "DF", "FH", "FL")]
        reg = A[0][1]
        fx = (A[0][0] + A[1][0]) / 2; fr = (A[2][0] + A[3][0]) / 2
    else:
        a, b = get9(k, "FIX", comb, o1), get9(k, "FREE", comb, o1)
        reg = a[1]; fx, fr = a[0], b[0]
    m = reg == region
    return fx[m], fr[m]

def fig10_rows(region):
    rows = []
    for k, comb, o1 in readings9():
        for how in ("mean", "pooled"):
            for w in ("E3", "E4", "E2"):
                fx, fr = fixed_free(k, comb, o1, region, how)
                x, y = fx[:, iw[w]], fr[:, iw[w]]
                pw = stats.wilcoxon(x, y).pvalue; pm = mwu(x, y)
                rows.append({"set": k, "comb": comb, "O1": o1, "fixed/free": how, "window end": w, "n": len(x),
                             "mean fixed": x.mean(), "mean free": y.mean(), "p Wilcoxon": pw, "p MWU": pm})
    return pd.DataFrame(rows)

In [ ]:
r10a = fig10_rows("DLS")
print(r10a.round(5).to_string())
main = r10a[r10a["window end"] != "E2"]
g = main.groupby(["set", "comb", "O1", "fixed/free"])
ok = g.apply(lambda d: bool(((d["mean fixed"] > d["mean free"]) & (d["p Wilcoxon"] < .05)).all()), include_groups=False).rename("Wilcoxon: fixed > free, both windows")
ok2 = g.apply(lambda d: bool(((d["mean fixed"] > d["mean free"]) & (d["p MWU"] < .05)).all()), include_groups=False).rename("MWU: fixed > free, both windows")
rows = []
for k, comb, o1 in readings9():
    cr, pk = {}, {}
    for typ in ("SF", "DF", "FH", "FL"):
        real, reg, avg, thr, _ = get9(k, typ, comb, o1)
        f, s = rise(avg["DLS"], thr["DLS"]); cr[typ] = T_END[s] if s is not None else np.nan
        m34 = (PH_END == 3) | (PH_END == 4)
        pk[typ] = avg["DLS"][:len(SLW)][m34].max()
    order = min(cr["SF"], cr["DF"]) < cr["FH"] < cr["FL"]
    rows.append({"set": k, "comb": comb, "O1": o1, **{f"rise {t}": cr[t] for t in cr}, **{f"peak P3-4 {t}": round(pk[t], 3) for t in pk},
                 "order SF/DF, FH, FL": order, "FL peak lowest": pk["FL"] == min(pk.values())})
ordr = pd.DataFrame(rows)
print(pd.concat([ok, ok2], axis=1).to_string())
print(ordr.to_string())

**Result: not reproduced.** The order of crossings fails under every reading: with the trials pooled FH rises first (2.15 s, before SF 2.85 and DF 2.75 to 2.80), and with MI per block the FL peak in Phases 3 and 4 is above the SF peak in every reading (bias-corrected, 0.088 against 0.079 bits/s). The fixed against free difference holds in both windows with the Wilcoxon test under the shuffle-corrected MI (pooled or per block) and under the bias-corrected MI with the trials pooled (mean of block types: E3 0.043 against 0.024, p = 0.039; E4 0.061 against 0.046, p = 0.029); it never holds with the Mann-Whitney test on the deposited PANs (bias-corrected, pooled, p = 0.10 and 0.20), and with bias-corrected MI per block E3 gives p = 0.29.

In [ ]:
r10b = fig10_rows("DMS")
print(r10b.round(5).to_string())
main = r10b[r10b["window end"] != "E2"]
g = main.groupby(["set", "comb", "O1", "fixed/free"])
print(pd.concat([g.apply(lambda d: bool(((d["mean fixed"] > d["mean free"]) & (d["p Wilcoxon"] < .05)).all()), include_groups=False).rename("Wilcoxon both"),
                 g.apply(lambda d: bool(((d["mean fixed"] > d["mean free"]) & (d["p MWU"] < .05)).all()), include_groups=False).rename("MWU both")], axis=1).to_string())

**Result: not reproduced.** In no reading is DMS action information above free-choice blocks with p < 0.05 in both windows. At the window ending at E4 fixed exceeds free (reading c, pooled: 0.070 against 0.038, Wilcoxon p = 0.0003), but at the window ending at E3 the difference is not significant (0.017 against 0.007, p = 0.079; per block p = 0.20 to 0.22).

In [ ]:
r10c = fig10_rows("VS")
print(r10c.round(5).to_string())
main = r10c[r10c["window end"] != "E2"]
g = main.groupby(["set", "comb", "O1", "fixed/free"])
print(pd.concat([g.apply(lambda d: bool((d["p Wilcoxon"] < .05).any()), include_groups=False).rename("Wilcoxon: a window p < 0.05"),
                 g.apply(lambda d: bool((d["p MWU"] < .05).any()), include_groups=False).rename("MWU: a window p < 0.05")], axis=1).to_string())
rows = []
for k, comb, o1 in readings9():
    row = {"set": k, "comb": comb, "O1": o1}
    for typ in ("SF", "DF", "FH", "FL"):
        real, reg, avg, thr, _ = get9(k, typ, comb, o1)
        m3 = PH_END == 3
        a = avg["VS"][:len(SLW)][m3]; t = thr["VS"][:len(SLW)][m3]
        row[f"{typ} P3 bins above"] = int((a > t).sum()); row[f"{typ} P3 peak"] = round(a.max(), 3)
    rows.append(row)
print("VS region average in Phase 3 (window ends in Phase 3):")
print(pd.DataFrame(rows).to_string())

**Result: blocked (methods_underspecified).** The absence holds when the last 20 trials of the blocks of a type are pooled: no window gives p < 0.05 under either test or any MI reading (bias-corrected, Wilcoxon p = 0.77 and 0.63). With MI per block it does not: the Wilcoxon test finds fixed above free in the window ending at E3 (bias-corrected 0.066 against 0.053 bits/s, p = 0.024). The paper does not say how the blocks of a type are combined. The VS region average passes its threshold in Phase 3 in all four block types when pooled (17 to 28 windows each). With MI per block it does so in 1 window for SF, 17 for DF, 3 to 4 for FH and 0 to 2 for FL, so a peak specific to SF blocks is not seen.

## fig10d-dls-stay-above-switch

> In free-choice blocks, DLS action information is stronger in stay trials than in switch
> trials in Phase 3, while DMS shows no difference between stay and switch trials.

Stay: the action repeats that of the previous choice trial of the session; switch: it does
not. Trials: (a) the last 20 trials of each free-choice block, (b) all free-choice trials. MI
pooled over the selected trials per neuron, windows ending at E3 and E4, Mann-Whitney U test.

In [ ]:
def stay_switch(nset, which, rng):
    out = []
    for n in nset:
        tr = n["tr"]; y = (tr.action == 1).to_numpy()
        allch = T[(T.session == n["session"]) & T.choice].sort_values("trial")
        prev = dict(zip(allch.trial.to_numpy()[1:], allch.action.to_numpy()[:-1]))
        pa = np.array([prev.get(t, np.nan) for t in tr.trial])
        stay = pa == tr.action.to_numpy()
        sw = np.isfinite(pa) & ~stay
        if which == "last20":
            sel = np.zeros(len(tr), bool)
            for blks in (blocks_last20(n).get("FREE", []),):
                for bi in blks:
                    sel[bi] = True
        else:
            sel = (tr.btype == "FREE").to_numpy()
        C = S.wcounts(n, SPW)
        r = {}
        for nm, m in (("stay", sel & stay), ("switch", sel & sw)):
            res = S.mi_with_shuffles(C[m], y[m], rng, 100, 0.5)
            r[nm] = (res["plugin"][0], res["plugin"][1].mean(0), int(m.sum()), res["pt"][0])
        out.append(r)
    return out
SS = {(k, w): stay_switch(v, w, np.random.default_rng(SEED + 1000 + 10 * (k != "deposited PANs") + (w == "all"))) for k, v in SETS.items() for w in ("last20", "all")}

In [ ]:
rows = []
for (k, w), res in SS.items():
    reg = REG[k]
    for o1 in ("a", "b", "c"):
        for region in ("DLS", "DMS"):
            m = reg == region
            def val(r, wi):
                return r[3][wi] if o1 == "c" else r[0][wi] - (r[1][wi] if o1 == "b" else 0)
            for wi, wn in ((1, "E3"), (2, "E4")):
                st = np.array([val(r["stay"], wi) for r in res])[m]
                sw = np.array([val(r["switch"], wi) for r in res])[m]
                rows.append({"set": k, "trials": w, "O1": o1, "region": region, "window end": wn,
                             "mean stay": st.mean(), "mean switch": sw.mean(), "p MWU": mwu(st, sw)})
r10d = pd.DataFrame(rows)
print(r10d.round(5).to_string())
ns = pd.DataFrame([{"stay": r["stay"][2], "switch": r["switch"][2]} for r in SS[("deposited PANs", "last20")]])
na = pd.DataFrame([{"stay": r["stay"][2], "switch": r["switch"][2]} for r in SS[("deposited PANs", "all")]])
print("Trials per neuron (median), last 20:", ns.median().to_dict(), "; all free-choice:", na.median().to_dict())
hold = r10d.groupby(["set", "trials", "O1"]).apply(lambda d: bool(
    ((d.region == "DLS") & (d["window end"] == "E3") & (d["mean stay"] > d["mean switch"]) & (d["p MWU"] < .05)).any()
    and not ((d.region == "DMS") & (d["p MWU"] < .05)).any()), include_groups=False)
print(hold.rename("holds").to_string())

**Result: reproduced, on the adopted reading.** With bias-corrected MI, DLS action information in the window ending at E3 is higher in stay than in switch trials (last 20 trials: 0.031 against 0.008 bits/s, p = 2e-5; all free-choice trials: 0.028 against 0.008, p = 0.00014), and DMS shows no difference in either window (p 0.18 to 0.72). With plug-in MI, which is biased upward for the fewer switch trials (median 66 against 152 trials per neuron), DLS shows no difference (p = 0.18 to 0.94) and, with the last 20 trials, DMS switch exceeds stay (p = 0.005 and 0.036).

## Summary

| Claim | Outcome | Rests on an assumption |
|---|---|---|
| trial-counts | reproduced | no |
| trials-per-block | reproduced | no |
| fig2c-extinction-differs | reproduced | no |
| fig2b-extinction-time-course | reproduced | no |
| fig2d-f-decision-trees | reproduced | no |
| fig2g-fq-variable-best | not reproduced | yes (model forms and priors) |
| actor-critic-near-chance | reproduced | no |
| neuron-counts | not reproduced | no |
| fig4b-vs-phase1 | reproduced | no |
| fig4b-dms-phase5 | reproduced | yes (smoothing, trials) |
| fig4b-vs-phase7 | reproduced | yes (activation reading) |
| fig4c-activation-duration | partially reproduced | yes (smoothing, trials) |
| fig5b-state-info-vs | partially reproduced | yes (MI bias correction, Phase 1 mean) |
| fig5e-action-info-rises-dls | not reproduced | no |
| fig5e-action-info-cue-offset-dls | reproduced | yes (MI bias correction) |
| fig5e-action-info-center-exit | reproduced | no |
| fig5d-action-info-phase5-dms | reproduced | no |
| fig5g-reward-info | blocked (methods_underspecified) | |
| fig6-event-coding-proportions | partially reproduced | yes (trials) |
| fig6-above-chance | reproduced | no |
| fig6bc-similar-fixed-free | partially reproduced | no |
| fig8a-d-action-reward-coding | partially reproduced | yes (trials, window axis) |
| fig8e-action-value-dms | reproduced | yes (trials, window axis) |
| fig8e-qr-dms | not reproduced | yes (trials, window axis) |
| fig8f-state-value-vs | partially reproduced | yes (trials, window axis) |
| no-policy-coding | reproduced | no |
| fig9-temporal-order-all-blocks | not reproduced | no |
| fig9-dms-never-above-dls | reproduced | yes (MI bias correction) |
| fig10a-dls-fixed-above-free | not reproduced | no |
| fig10b-dms-fixed-above-free | not reproduced | no |
| fig10c-vs-no-fixed-free-difference | blocked (methods_underspecified) | |
| fig10d-dls-stay-above-switch | reproduced | yes (MI bias correction) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The neural claims use the units the dandiset
flags as PANs (204, 112, 138), not a set rebuilt from the Methods, which cannot apply the
waveform and histology conditions; the rebuilt set is shown as a variant. The choice models
use 86 sessions where the paper used 202, and their equations, which are missing from the
paper's text, follow the claims file's reading. Where the paper leaves a method open and the
readings disagree, these choices were adopted: extinction trials excluded from "whole trials"
(the Methods list single fixed, double fixed and free-choice blocks); EASHs smoothed with a 30
ms Gaussian (the Fig. 3 legend); "reached 0.5 during each trial phase" read as some bin at or
above 0.5; the mutual information corrected for bias by Panzeri and Treves (1996), whom the
paper cites for it; the Fig. 8 regression on free-choice trials (the trials of the FQ model)
with windows on the EASH axis (the axis of the Fig. 7 regression bins).

**What was found about the dataset.** The trial table's `state` codes are described as 1 =
left, 2 = right, 3 = choice tone, but the data show 2 = left, 3 = right and 1 = choice tone.
The dandiset holds 859 units, more than the paper's stable neurons, with a `PANs` flag that
does not follow the Methods' ISI rule alone (174 units pass the rule without the flag) and
with no record of the stability and duplicate exclusions; 15 CS units carry the flag though
the paper analyses only DLS, DMS and VS. One file (LE115_14) has behaviour but no units, so
there are 87 files for the paper's 86 recording sessions. Some events are out of the task's
stated range (cue durations from 0.009 s where the paper gives 1 to 2 s). The behavioural
model comparison of 202 sessions cannot be repeated, as the dandiset holds 86.